In [1]:

from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

# Search for existing audit reports, manifests, and label mappings.
keywords = [
    "audit", "manifest", "label", "phenotype",
    "hf_lung", "fraiwan", "icbhi"
]

matches = []
for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue
    if any(k in path.name.lower() for k in keywords):
        if ".git" not in path.parts and "__pycache__" not in path.parts:
            matches.append(path)

print(f"Found {len(matches)} relevant files\n")
for path in matches[:150]:
    print(path.relative_to(PROJECT_ROOT))

print("\n--- Existing ICBHI phenotype manifests ---")
manifest_dir = (
    PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline"
)
if manifest_dir.exists():
    for path in sorted(manifest_dir.glob("*")):
        if path.is_file():
            print(path.name, f"({path.stat().st_size:,} bytes)")

print("\n--- Existing phenotype feature table ---")
feature_path = (
    PROJECT_ROOT / "data/processed/features/"
    "respiratory_cycle_features.csv"
)
if feature_path.exists():
    df = pd.read_csv(feature_path, nrows=5)
    print("Columns:", df.columns.tolist())
    print(df.head(2).to_string(index=False))
else:
    print("Feature table not found at expected path.")


Found 9849 relevant files

Notebooks 2/01_Fraiwan_Dataset_Audit.ipynb
Notebooks 2/01_Data_Audit.ipynb
Notebooks 2/01_HF_Lung_V1_Dataset_Audit.ipynb
notebooks/19_ICBHI_Sound_Phenotype_Classification.ipynb
reports/experiment_manifest.json
reports/final_artifacts/model_package_manifest.json
reports/final_artifacts/comparison_manifest.json
reports/final_artifacts/final_submission_zip_manifest.csv
reports/final_artifacts/artifact_file_manifest.csv
reports/final_artifacts/model_package_manifest.csv
notebooks/.ipynb_checkpoints/19_ICBHI_Sound_Phenotype_Classification-checkpoint.ipynb
data/raw/HF Lung V1/HF_Lung_V1-master.zip
data/raw/HF Lung V1/HF_Lung_V1-master/Dataset/test/steth_20190821_12_00_25_label.txt
data/raw/HF Lung V1/HF_Lung_V1-master/Dataset/test/steth_20190810_14_21_08_label.txt
data/raw/HF Lung V1/HF_Lung_V1-master/Dataset/test/steth_20191017_10_28_26_label.txt
data/raw/HF Lung V1/HF_Lung_V1-master/Dataset/test/trunc_2019-08-21-10-35-06-L1_1_label.txt
data/raw/HF Lung V1/HF_Lung

In [2]:

from pathlib import Path
import json

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

audit_files = [
    PROJECT_ROOT / "Notebooks 2/01_Fraiwan_Dataset_Audit.ipynb",
    PROJECT_ROOT / "Notebooks 2/01_HF_Lung_V1_Dataset_Audit.ipynb",
]

for path in audit_files:
    print("\n" + "=" * 85)
    print("AUDIT:", path.name)
    print("Exists:", path.exists())
    print("=" * 85)

    if not path.exists():
        print("Check the exact path from the earlier search output.")
        continue

    nb = json.loads(path.read_text(encoding="utf-8"))

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        outputs = []

        for output in cell.get("outputs", []):
            if "text" in output:
                outputs.append("".join(output["text"]))
            elif "data" in output:
                data = output["data"]
                if "text/plain" in data:
                    outputs.append("".join(data["text/plain"]))
                elif "text/html" in data:
                    outputs.append("[HTML table output available]")

        # Show section headings and stored results, not every code cell.
        is_heading = (
            cell.get("cell_type") == "markdown"
            and source.strip().startswith("#")
        )

        if is_heading or outputs:
            if is_heading:
                print(f"\n{source.strip()[:180]}")
            for result in outputs:
                if result.strip():
                    print(f"\n[Cell {i} output]")
                    print(result[:1200])



AUDIT: 01_Fraiwan_Dataset_Audit.ipynb
Exists: True

[Cell 0 output]
Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification

Fraiwan folders found:
 - /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan
 - /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/FRAIWAN

Project folders:
 - .git
 - .pytest_cache
 - .ruff_cache
 - .venv
 - .vscode
 - Notebooks 2
 - app
 - data
 - docs
 - models
 - notebooks
 - reports
 - scripts
 - tests


[Cell 1 output]
========== RAW DATASET FOLDERS ==========

Fraiwan: exists=True
Resolved path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan
Same resolved location as other spelling: False

FRAIWAN: exists=True
Resolved path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/FRAIWAN
Same resolved location as other spelling: False

========== DATASET CONTENTS ==========
Dataset root: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan
Total files: 339

File counts by extension

In [4]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/vs/Sem 5/Respiratory-Disease-Classification")

MANIFEST_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
    / "cnn_baseline"
)

print("=" * 75)
print("STEP 1 — LOCATE ICBHI PHENOTYPE SPLIT MANIFESTS")
print("=" * 75)

if not MANIFEST_DIR.exists():
    raise FileNotFoundError(
        f"Manifest directory not found: {MANIFEST_DIR}"
    )

manifest_files = sorted(MANIFEST_DIR.glob("*.csv"))

if not manifest_files:
    raise FileNotFoundError(
        f"No CSV manifests found in {MANIFEST_DIR}"
    )

for file in manifest_files:
    print(f"\n{file.name} | {file.stat().st_size:,} bytes")

print("\n" + "=" * 75)
print("STEP 2 — INSPECT MANIFEST SCHEMAS")
print("=" * 75)

for file in manifest_files:
    df = pd.read_csv(file, nrows=3)
    print(f"\nFILE: {file.name}")
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

print("\n" + "=" * 75)
print("STEP 3 — FIND THE ACTUAL SPLIT AND LABEL COLUMNS")
print("=" * 75)

for file in manifest_files:
    df = pd.read_csv(file)

    split_candidates = [
        c for c in df.columns
        if any(term in c.lower() for term in
               ["split", "partition", "subset"])
    ]

    label_candidates = [
        c for c in df.columns
        if any(term in c.lower() for term in
               ["phenotype", "sound_label", "sound_type", "label", "class"])
    ]

    print(f"\nFILE: {file.name}")
    print("Rows:", len(df))
    print("Possible split columns:", split_candidates)
    print("Possible label columns:", label_candidates)

    for col in split_candidates:
        print(f"\nCounts for {col}:")
        print(df[col].value_counts(dropna=False).to_string())

    for col in label_candidates:
        print(f"\nCounts for {col}:")
        print(df[col].value_counts(dropna=False).to_string())

print("\nRead-only manifest inspection complete.")


STEP 1 — LOCATE ICBHI PHENOTYPE SPLIT MANIFESTS

cnn_baseline_experiment_summary.csv | 500 bytes

cnn_baseline_metrics.csv | 309 bytes

cnn_patient_disjoint_manifest.csv | 888,179 bytes

cnn_split_summary.csv | 71 bytes

cnn_test_predictions.csv | 223,653 bytes

cnn_training_class_weights.csv | 307 bytes

cnn_training_history.csv | 3,380 bytes

test_cnn_manifest.csv | 198,749 bytes

train_cnn_manifest.csv | 567,285 bytes

validation_cnn_manifest.csv | 122,343 bytes

STEP 2 — INSPECT MANIFEST SCHEMAS

FILE: cnn_baseline_experiment_summary.csv
Columns: ['experiment', 'input', 'input_shape', 'split_strategy', 'augmentation', 'class_weights', 'model_selection', 'early_stopping', 'test_patients', 'test_cycles', 'accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1']
                  experiment               input  input_shape   split_strategy                               augmentation class_weights model_selection  early_stopping  test_patients  test_

In [5]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/vs/Sem 5/Respiratory-Disease-Classification")

print("=" * 75)
print("STEP 4 — LOCATE ORIGINAL ICBHI ANNOTATIONS AND PHENOTYPE LABELS")
print("=" * 75)

search_roots = [
    PROJECT_ROOT / "data" / "raw" / "ICBHI",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "reports",
]

keywords = (
    "annotation",
    "cycle",
    "phenotype",
    "crackle",
    "wheeze",
    "respiratory_cycle",
    "manifest",
)

matches = []

for root in search_roots:
    print(f"\nSearching: {root}")
    if not root.exists():
        print("Directory not found.")
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if any(part.startswith(".") for part in path.parts):
            continue

        name = path.name.lower()

        if any(word in name for word in keywords):
            matches.append(path)

for path in sorted(set(matches)):
    print(f"{path.relative_to(PROJECT_ROOT)}")

print("\n" + "=" * 75)
print("STEP 5 — CHECK EXISTING PHENOTYPE FEATURE TABLE")
print("=" * 75)

feature_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "features"
    / "respiratory_cycle_features.csv"
)

if feature_path.exists():
    features = pd.read_csv(feature_path)

    print("File:", feature_path.relative_to(PROJECT_ROOT))
    print("Shape:", features.shape)
    print("Columns:")
    print(features.columns.tolist())

    label_columns = [
        col for col in features.columns
        if any(term in col.lower() for term in [
            "crackle", "wheeze", "phenotype", "label",
            "patient", "recording", "cycle", "audio", "path"
        ])
    ]

    print("\nRelevant columns:")
    print(label_columns)

    if label_columns:
        print("\nSample rows:")
        print(features[label_columns].head(5).to_string(index=False))
else:
    print("Existing feature table not found at the expected path.")

print("\nNo files were modified.")


STEP 4 — LOCATE ORIGINAL ICBHI ANNOTATIONS AND PHENOTYPE LABELS

Searching: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI

Searching: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed

Searching: /Users/vs/Sem 5/Respiratory-Disease-Classification/reports
data/processed/analysis/crackle_feature_correlations.csv
data/processed/analysis/crackle_feature_means.csv
data/processed/analysis/wheeze_feature_correlations.csv
data/processed/analysis/wheeze_feature_means.csv
data/processed/cycles_clean.csv
data/processed/deep_learning/cnn_baseline/cnn_patient_disjoint_manifest.csv
data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv
data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv
data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv
data/processed/deep_learning/deep_learning_manifest.csv
data/processed/deep_learning/test_manifest.csv
data/processed/deep_learning/train_manifest.csv
data/processed/features/respiratory_cyc

In [6]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/Users/vs/Sem 5/Respiratory-Disease-Classification")

FEATURE_PATH = (
    PROJECT_ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)

features = pd.read_csv(FEATURE_PATH)

# Map the verified binary annotations to four phenotype classes.
def map_phenotype(row):
    crackles = row["crackles"]
    wheezes = row["wheezes"]

    if pd.isna(crackles) or pd.isna(wheezes):
        return np.nan

    crackles = int(crackles)
    wheezes = int(wheezes)

    mapping = {
        (0, 0): "Normal",
        (1, 0): "Crackles",
        (0, 1): "Wheezes",
        (1, 1): "Both",
    }

    return mapping.get((crackles, wheezes), np.nan)


features["phenotype_label"] = features.apply(map_phenotype, axis=1)

# Resolve audio paths without changing the original feature table.
def resolve_audio_path(value):
    if pd.isna(value):
        return None

    path = Path(str(value))
    candidates = [
        path if path.is_absolute() else PROJECT_ROOT / "data" / "processed" / path,
        PROJECT_ROOT / "data" / "processed" / "audio" / path.name,
    ]

    for candidate in candidates:
        if candidate.is_file():
            return str(candidate.resolve())

    return None


features["resolved_audio_path"] = features["processed_path"].apply(
    resolve_audio_path
)

print("=" * 75)
print("ICBHI PHENOTYPE LABEL VALIDATION")
print("=" * 75)

print("Total cycles:", len(features))
print("\nRaw annotation combinations:")
print(
    features.groupby(["crackles", "wheezes"], dropna=False)
    .size()
    .rename("cycles")
    .to_string()
)

print("\nMapped phenotype counts:")
print(
    features["phenotype_label"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nMapped phenotype counts by patient:")
print(
    features.groupby("phenotype_label", dropna=False)["patient_id"]
    .nunique()
    .to_string()
)

print("\nMissing phenotype labels:", features["phenotype_label"].isna().sum())
print("Missing patient IDs:", features["patient_id"].isna().sum())
print("Missing processed paths:", features["processed_path"].isna().sum())
print("Audio paths successfully resolved:",
      features["resolved_audio_path"].notna().sum())
print("Audio paths not resolved:",
      features["resolved_audio_path"].isna().sum())

print("\nSample unresolved paths:")
print(
    features.loc[
        features["resolved_audio_path"].isna(),
        ["patient_id", "recording_id", "processed_path"]
    ].head(10).to_string(index=False)
)

print("\nSample mapped records:")
print(
    features[
        [
            "patient_id",
            "recording_id",
            "crackles",
            "wheezes",
            "phenotype_label",
            "resolved_audio_path",
        ]
    ].head(10).to_string(index=False)
)

print("\nValidation complete. No files were modified.")


ICBHI PHENOTYPE LABEL VALIDATION
Total cycles: 6898

Raw annotation combinations:
crackles  wheezes
0         0          3642
          1           886
1         0          1864
          1           506

Mapped phenotype counts:
phenotype_label
Normal      3642
Crackles    1864
Wheezes      886
Both         506

Mapped phenotype counts by patient:
phenotype_label
Both         35
Crackles     74
Normal      124
Wheezes      63

Missing phenotype labels: 0
Missing patient IDs: 0
Missing processed paths: 0
Audio paths successfully resolved: 6898
Audio paths not resolved: 0

Sample unresolved paths:
Empty DataFrame
Columns: [patient_id, recording_id, processed_path]
Index: []

Sample mapped records:
 patient_id recording_id  crackles  wheezes phenotype_label                                                                               resolved_audio_path
        101          1b1         0        0          Normal /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycl

In [8]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/vs/Sem 5/Respiratory-Disease-Classification")

FEATURE_PATH = (
    PROJECT_ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)

SPLIT_PATH = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "cnn_baseline" / "cnn_patient_disjoint_manifest.csv"
)

features = pd.read_csv(FEATURE_PATH)
splits = pd.read_csv(SPLIT_PATH)

join_keys = ["patient_id", "recording_id", "chest_location"]

print("=" * 70)
print("1. DUPLICATE COUNTS")
print("=" * 70)

print("Feature rows:", len(features))
print("Split rows:", len(splits))
print("Duplicate feature keys:", features.duplicated(join_keys).sum())
print("Duplicate split keys:", splits.duplicated(join_keys).sum())

print("\n" + "=" * 70)
print("2. EXAMPLE DUPLICATE SPLIT KEYS")
print("=" * 70)

duplicate_mask = splits.duplicated(join_keys, keep=False)
duplicate_rows = splits.loc[duplicate_mask].sort_values(join_keys)

print(duplicate_rows.head(20).to_string(index=False))

print("\n" + "=" * 70)
print("3. CHECK WHETHER DUPLICATE KEYS HAVE DIFFERENT SPLITS")
print("=" * 70)

if duplicate_mask.any():
    grouped = (
        splits.loc[duplicate_mask]
        .groupby(join_keys, dropna=False)
        .agg(
            rows=("cnn_split", "size"),
            distinct_splits=("cnn_split", "nunique"),
            splits=("cnn_split", lambda x: ", ".join(sorted(x.astype(str).unique()))),
            distinct_paths=("audio_path", "nunique"),
        )
        .reset_index()
    )
    print(grouped.head(20).to_string(index=False))
else:
    print("No duplicate keys found.")

print("\n" + "=" * 70)
print("4. AVAILABLE CYCLE IDENTIFIERS")
print("=" * 70)

print("Feature columns:")
print(features.columns.tolist())

print("\nSplit manifest columns:")
print(splits.columns.tolist())

print("\nFeature sample:")
print(features[join_keys + ["cycle_number", "processed_path"]].head(10).to_string(index=False))

print("\nSplit sample:")
print(splits.head(10).to_string(index=False))

print("\nInspection only — no files were modified.")


1. DUPLICATE COUNTS
Feature rows: 6898
Split rows: 6898
Duplicate feature keys: 5978
Duplicate split keys: 5978

2. EXAMPLE DUPLICATE SPLIT KEYS
 patient_id recording_id chest_location diagnosis  disease_label official_split                                                                                        audio_path cnn_split
        101          1b1             Al      URTI              7           test /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_001.wav      test
        101          1b1             Al      URTI              7           test /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_002.wav      test
        101          1b1             Al      URTI              7           test /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/101/1b1_Al_003.wav      test
        101          1b1             Al      URTI              7           test /Users/vs/Sem 5/Respira

In [9]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/vs/Sem 5/Respiratory-Disease-Classification")

FEATURE_PATH = (
    PROJECT_ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)
SPLIT_PATH = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "cnn_baseline" / "cnn_patient_disjoint_manifest.csv"
)

features = pd.read_csv(FEATURE_PATH)
splits = pd.read_csv(SPLIT_PATH)

# Normalize paths to a common absolute representation.
features["audio_key"] = features["processed_path"].apply(
    lambda p: str((PROJECT_ROOT / "data" / "processed" / str(p)).resolve())
)
splits["audio_key"] = splits["audio_path"].apply(
    lambda p: str(Path(str(p)).resolve())
)

# Confirm that each cycle is uniquely represented by its audio path.
assert features["audio_key"].is_unique, (
    "Feature table contains duplicate audio paths."
)
assert splits["audio_key"].is_unique, (
    "Split manifest contains duplicate audio paths."
)

# Validate exact path matching before merging.
feature_paths = set(features["audio_key"])
split_paths = set(splits["audio_key"])

print("Feature paths:", len(feature_paths))
print("Split paths:", len(split_paths))
print("Feature paths missing from split manifest:",
      len(feature_paths - split_paths))
print("Split paths missing from feature table:",
      len(split_paths - feature_paths))

if feature_paths != split_paths:
    print("\nExample feature paths absent from split:")
    print(list(feature_paths - split_paths)[:5])
    print("\nExample split paths absent from features:")
    print(list(split_paths - feature_paths)[:5])
    raise ValueError("Audio paths do not match exactly; investigate before merging.")

# Attach the existing patient-disjoint CNN split.
split_lookup = splits[["audio_key", "cnn_split"]].copy()

icbhi_manifest = features.merge(
    split_lookup,
    on="audio_key",
    how="left",
    validate="one_to_one"
)

assert len(icbhi_manifest) == len(features)
assert icbhi_manifest["cnn_split"].notna().all()

# Derive the four-class sound phenotype from the existing annotations.
def map_phenotype(row):
    crackles = int(row["crackles"])
    wheezes = int(row["wheezes"])

    if crackles == 0 and wheezes == 0:
        return "Normal"
    if crackles == 1 and wheezes == 0:
        return "Crackles"
    if crackles == 0 and wheezes == 1:
        return "Wheezes"
    if crackles == 1 and wheezes == 1:
        return "Both"
    return None

icbhi_manifest["phenotype_label"] = icbhi_manifest.apply(
    map_phenotype, axis=1
)

assert icbhi_manifest["phenotype_label"].notna().all()

# Check that no patient occurs in more than one split.
patient_split_counts = (
    icbhi_manifest.groupby("patient_id")["cnn_split"].nunique()
)
leaking_patients = patient_split_counts[patient_split_counts > 1]

print("\nManifest rows:", len(icbhi_manifest))
print("\nPhenotype counts:")
print(icbhi_manifest["phenotype_label"].value_counts())

print("\nSplit counts:")
print(icbhi_manifest["cnn_split"].value_counts())

print("\nPhenotype by split:")
print(pd.crosstab(
    icbhi_manifest["cnn_split"],
    icbhi_manifest["phenotype_label"]
))

print("\nPatients appearing in multiple splits:", len(leaking_patients))
assert len(leaking_patients) == 0, "Patient leakage detected!"

print("\nICBHI manifest validated. No files saved or modified.")


Feature paths: 6898
Split paths: 6898
Feature paths missing from split manifest: 0
Split paths missing from feature table: 0

Manifest rows: 6898

Phenotype counts:
phenotype_label
Normal      3642
Crackles    1864
Wheezes      886
Both         506
Name: count, dtype: int64

Split counts:
cnn_split
train         4423
test          1561
validation     914
Name: count, dtype: int64

Phenotype by split:
phenotype_label  Both  Crackles  Normal  Wheezes
cnn_split                                       
test              171       462     705      223
train             316      1219    2342      546
validation         19       183     595      117

Patients appearing in multiple splits: 0

ICBHI manifest validated. No files saved or modified.


In [10]:

from pathlib import Path

OUTPUT_DIR = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "combined_phenotype"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ICBHI_OUTPUT = OUTPUT_DIR / "icbhi_phenotype_manifest.csv"

# Keep useful metadata and the cycle-level audio path.
save_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "phenotype_label",
    "processed_path",
    "audio_key",
    "cnn_split",
]

icbhi_manifest[save_columns].to_csv(ICBHI_OUTPUT, index=False)

print("Saved:", ICBHI_OUTPUT)
print("Rows:", len(icbhi_manifest))
print("Columns:", len(save_columns))
print("File exists:", ICBHI_OUTPUT.exists())


Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_phenotype/icbhi_phenotype_manifest.csv
Rows: 6898
Columns: 14
File exists: True


In [11]:

from pathlib import Path
from collections import Counter

HF_ROOT = (
    PROJECT_ROOT / "data" / "raw" / "HF Lung V1"
    / "HF_Lung_V1-master"
)

assert HF_ROOT.exists(), f"HF dataset not found: {HF_ROOT}"

print("HF dataset root:", HF_ROOT)

# Show directory structure without dumping the entire dataset.
for path in sorted(HF_ROOT.iterdir()):
    if path.is_dir():
        print(f"\n[{path.name}/]")
        children = sorted(path.iterdir())
        print("Immediate entries:", len(children))
        for child in children[:8]:
            print("  ", child.name)
        if len(children) > 8:
            print("   ...")
    else:
        print(path.name)

# Count WAV/TXT files by their immediate parent directory.
counts = Counter()
for path in HF_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".wav", ".txt"}:
        counts[(path.parent.relative_to(HF_ROOT).as_posix(), path.suffix.lower())] += 1

print("\nAudio and annotation counts by directory:")
for key, count in sorted(counts.items()):
    print(f"{key}: {count}")

print("\nNo files modified.")


HF dataset root: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/HF Lung V1/HF_Lung_V1-master
.DS_Store

[Dataset/]
Immediate entries: 3
   .DS_Store
   test
   train
Disclaimer
LICENSE
README.md

Audio and annotation counts by directory:
('Dataset/test', '.txt'): 1956
('Dataset/test', '.wav'): 1956
('Dataset/train', '.txt'): 7809
('Dataset/train', '.wav'): 7809

No files modified.


In [12]:

from pathlib import Path
from collections import Counter

HF_ROOT = (
    PROJECT_ROOT / "data" / "raw" / "HF Lung V1"
    / "HF_Lung_V1-master"
)

DATASET_ROOT = HF_ROOT / "Dataset"
TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"

# 1. Read the dataset README.
readme_path = HF_ROOT / "README.md"

print("=" * 75)
print("README")
print("=" * 75)

if readme_path.exists():
    print(readme_path.read_text(encoding="utf-8", errors="replace")[:12000])
else:
    print("README.md not found at expected location.")

# 2. Inspect representative annotation files from each official split.
print("\n" + "=" * 75)
print("SAMPLE ANNOTATION FILES")
print("=" * 75)

for split_name, split_dir in [
    ("train", TRAIN_DIR),
    ("test", TEST_DIR),
]:
    txt_files = sorted(split_dir.glob("*.txt"))
    print(f"\n{split_name.upper()}: {len(txt_files)} TXT files")

    for txt_path in txt_files[:4]:
        print(f"\n--- {txt_path.name} ---")
        content = txt_path.read_text(
            encoding="utf-8", errors="replace"
        )
        print(repr(content[:1800]))

# 3. Count annotation line patterns across the dataset.
print("\n" + "=" * 75)
print("ANNOTATION LINE-FORMAT AUDIT")
print("=" * 75)

line_counts = Counter()
empty_files = Counter()
malformed_examples = []

for split_name, split_dir in [
    ("train", TRAIN_DIR),
    ("test", TEST_DIR),
]:
    for txt_path in split_dir.glob("*.txt"):
        content = txt_path.read_text(
            encoding="utf-8", errors="replace"
        )
        lines = [line.strip() for line in content.splitlines() if line.strip()]

        if not lines:
            empty_files[split_name] += 1
            continue

        for line in lines:
            parts = line.replace(",", " ").split()
            line_counts[(split_name, len(parts))] += 1

            # Expected interval-style annotations usually have
            # start time, end time, and one or more label fields.
            if len(parts) < 3 and len(malformed_examples) < 10:
                malformed_examples.append(
                    (split_name, txt_path.name, line)
                )

print("Non-empty annotation line counts by split and number of fields:")
for key, count in sorted(line_counts.items()):
    print(f"{key}: {count}")

print("\nEmpty annotation files:", dict(empty_files))
print("Examples of lines with fewer than 3 fields:")
for item in malformed_examples:
    print(item)

print("\nInspection only — no files modified.")


README
# HF_Lung_V1

This repository contains an open access lung sound database used for developing automated inhalation, exhalation, and adventitious sounds detection algorithms. 

The lung sound recordings come from two sources. The first source was a database used in a datathon in Taiwan Smart Emergency and Critical Care (TSECC), 2020, under the license of Creative Commons Attribution 4.0 (CC BY 4.0), provided by the Taiwan Society of Emergency and Critical Care Medicine (TSECCM). Lung sound recordings in the TSECC database were acquired from 261 patients. 

The second source was sound recordings acquired from 18 residents of a respiratory care ward (RCW) or a respiratory care center (RCC) in Northern Taiwan between August 2018 and October 2019. The recordings were approved by the Research Ethics Review Committee of Far Eastern Memorial Hospital (case number: 107052-F). Written informed consent was obtained from the 18 patients.

We have used the database to train and benchmark eig

In [13]:

from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

HF_ROOT = (
    PROJECT_ROOT / "data" / "raw" / "HF Lung V1"
    / "HF_Lung_V1-master"
)
DATASET_ROOT = HF_ROOT / "Dataset"

annotation_counts = Counter()
split_file_counts = Counter()
missing_audio = []
missing_annotation = []
invalid_lines = []
invalid_intervals = []
empty_annotations = []
recording_rows = []

def timestamp_to_seconds(value):
    parts = value.strip().split(":")
    if len(parts) != 3:
        raise ValueError(f"Unexpected timestamp: {value}")
    return (
        float(parts[0]) * 3600
        + float(parts[1]) * 60
        + float(parts[2])
    )

for split_name in ["train", "test"]:
    split_dir = DATASET_ROOT / split_name
    wav_lookup = {p.stem: p for p in split_dir.glob("*.wav")}
    txt_lookup = {
        p.stem.removesuffix("_label"): p
        for p in split_dir.glob("*.txt")
    }

    for stem in sorted(set(wav_lookup) | set(txt_lookup)):
        wav_path = wav_lookup.get(stem)
        txt_path = txt_lookup.get(stem)

        if wav_path is None:
            missing_audio.append((split_name, stem))
            continue
        if txt_path is None:
            missing_annotation.append((split_name, stem))
            continue

        content = txt_path.read_text(
            encoding="utf-8", errors="replace"
        )
        lines = [line.strip() for line in content.splitlines() if line.strip()]

        if not lines:
            empty_annotations.append((split_name, stem))

        valid_line_count = 0
        event_labels = set()

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            if len(parts) != 3:
                invalid_lines.append(
                    (split_name, txt_path.name, line_number, line)
                )
                continue

            label, start_text, end_text = parts
            annotation_counts[(split_name, label)] += 1
            event_labels.add(label)

            try:
                start = timestamp_to_seconds(start_text)
                end = timestamp_to_seconds(end_text)

                if start < 0 or end <= start or end > 15.05:
                    invalid_intervals.append(
                        (split_name, txt_path.name, line_number,
                         label, start, end)
                    )
                    continue
            except Exception as exc:
                invalid_intervals.append(
                    (split_name, txt_path.name, line_number,
                     label, str(exc))
                )
                continue

            valid_line_count += 1

        recording_rows.append({
            "split": split_name,
            "recording_id": stem,
            "audio_path": str(wav_path),
            "annotation_path": str(txt_path),
            "is_empty_annotation": len(lines) == 0,
            "valid_annotation_lines": valid_line_count,
            "annotation_labels": ",".join(sorted(event_labels)),
        })

print("ANNOTATION COUNTS BY SPLIT AND LABEL")
for (split_name, label), count in sorted(annotation_counts.items()):
    print(f"{split_name:5s} | {label:12s} | {count:6d}")

print("\nRECORDING / FILE CHECKS")
print("Matched recordings:", len(recording_rows))
print("Missing WAV files:", len(missing_audio))
print("Missing TXT files:", len(missing_annotation))
print("Empty annotation files:", len(empty_annotations))
print("Malformed annotation lines:", len(invalid_lines))
print("Invalid intervals:", len(invalid_intervals))

print("\nEmpty annotation examples:", empty_annotations[:10])
print("Malformed line examples:", invalid_lines[:5])
print("Invalid interval examples:", invalid_intervals[:5])

hf_annotation_audit = pd.DataFrame(recording_rows)

print("\nRecording audit shape:", hf_annotation_audit.shape)
print(hf_annotation_audit.head(5).to_string(index=False))

print("\nAudit only — no files modified.")


ANNOTATION COUNTS BY SPLIT AND LABEL
test  | D            |   1812
test  | E            |   2748
test  | I            |   6872
test  | Rhonchi      |    960
test  | Stridor      |     29
test  | Wheeze       |   1430
train | D            |  13794
train | E            |  15601
train | I            |  27223
train | Rhonchi      |   3780
train | Stridor      |    657
train | Wheeze       |   7027

RECORDING / FILE CHECKS
Matched recordings: 9765
Missing WAV files: 0
Missing TXT files: 0
Empty annotation files: 58
Malformed annotation lines: 0
Invalid intervals: 0

Empty annotation examples: [('train', 'steth_20190429_12_52_37'), ('train', 'steth_20190630_11_11_13'), ('train', 'trunc_2019-05-07-13-57-26-L8_2'), ('train', 'trunc_2019-05-07-14-29-34-L2_1'), ('train', 'trunc_2019-05-07-14-29-34-L2_14'), ('train', 'trunc_2019-05-07-14-29-34-L4_1'), ('train', 'trunc_2019-05-07-14-29-34-L5_14'), ('train', 'trunc_2019-05-07-14-29-34-L6_1'), ('train', 'trunc_2019-05-07-15-00-52-L4_3'), ('train', '

In [14]:

from pathlib import Path
import pandas as pd

HF_ROOT = (
    PROJECT_ROOT / "data" / "raw" / "HF Lung V1"
    / "HF_Lung_V1-master"
)
DATASET_ROOT = HF_ROOT / "Dataset"

def time_to_seconds(value):
    hours, minutes, seconds = value.split(":")
    return int(hours) * 3600 + int(minutes) * 60 + float(seconds)

def overlaps(a_start, a_end, b_start, b_end):
    # Strictly positive overlap; touching boundaries alone do not count.
    return min(a_end, b_end) > max(a_start, b_start)

candidate_rows = []
excluded_rows = []
annotation_totals = {}

for split_name in ["train", "test"]:
    split_dir = DATASET_ROOT / split_name

    for wav_path in sorted(split_dir.glob("*.wav")):
        txt_path = split_dir / f"{wav_path.stem}_label.txt"

        if not txt_path.exists():
            excluded_rows.append({
                "split": split_name,
                "recording_id": wav_path.stem,
                "reason": "missing_annotation",
            })
            continue

        raw_lines = [
            line.strip()
            for line in txt_path.read_text(
                encoding="utf-8", errors="replace"
            ).splitlines()
            if line.strip()
        ]

        if not raw_lines:
            excluded_rows.append({
                "split": split_name,
                "recording_id": wav_path.stem,
                "reason": "empty_annotation",
            })
            continue

        events = []
        for line in raw_lines:
            label, start_text, end_text = line.split()
            events.append({
                "label": label,
                "start": time_to_seconds(start_text),
                "end": time_to_seconds(end_text),
            })

        phases = [e for e in events if e["label"] in {"I", "E"}]
        if not phases:
            excluded_rows.append({
                "split": split_name,
                "recording_id": wav_path.stem,
                "reason": "no_respiratory_phase",
            })
            continue

        for phase_index, phase in enumerate(phases, start=1):
            phase_start, phase_end = phase["start"], phase["end"]

            overlapping_labels = {
                e["label"]
                for e in events
                if e["label"] not in {"I", "E"}
                and overlaps(
                    phase_start, phase_end, e["start"], e["end"]
                )
            }

            # Other adventitious sounds make this window unsuitable
            # for the current four-class phenotype mapping.
            if {"Rhonchi", "Stridor"} & overlapping_labels:
                excluded_rows.append({
                    "split": split_name,
                    "recording_id": wav_path.stem,
                    "phase_index": phase_index,
                    "start_time": phase_start,
                    "end_time": phase_end,
                    "reason": "rhonchi_or_stridor_overlap",
                })
                continue

            has_crackles = "D" in overlapping_labels
            has_wheeze = "Wheeze" in overlapping_labels

            if has_crackles and has_wheeze:
                phenotype = "Both"
            elif has_crackles:
                phenotype = "Crackles"
            elif has_wheeze:
                phenotype = "Wheezes"
            else:
                phenotype = "Normal"

            candidate_rows.append({
                "dataset": "HF_Lung_V1",
                "recording_id": wav_path.stem,
                "split": split_name,
                "audio_path": str(wav_path),
                "annotation_path": str(txt_path),
                "phase_index": phase_index,
                "phase_type": phase["label"],
                "start_time": phase_start,
                "end_time": phase_end,
                "duration_seconds": phase_end - phase_start,
                "crackles": int(has_crackles),
                "wheezes": int(has_wheeze),
                "phenotype_label": phenotype,
                "overlapping_event_labels": ",".join(
                    sorted(overlapping_labels)
                ),
            })

hf_window_preview = pd.DataFrame(candidate_rows)
hf_exclusion_preview = pd.DataFrame(excluded_rows)

print("CANDIDATE PHASE WINDOWS")
print("Total candidates:", len(hf_window_preview))
if not hf_window_preview.empty:
    print("\nClass counts:")
    print(hf_window_preview["phenotype_label"].value_counts())
    print("\nCounts by original split:")
    print(pd.crosstab(
        hf_window_preview["split"],
        hf_window_preview["phenotype_label"]
    ))
    print("\nPhase duration summary (seconds):")
    print(hf_window_preview["duration_seconds"].describe())

print("\nEXCLUSIONS")
if not hf_exclusion_preview.empty:
    print(hf_exclusion_preview["reason"].value_counts())
else:
    print("No exclusions.")

print("\nExample candidate windows:")
if not hf_window_preview.empty:
    print(hf_window_preview.head(10).to_string(index=False))

print("\nManifest preview only — no audio or manifest files saved.")


CANDIDATE PHASE WINDOWS
Total candidates: 47388

Class counts:
phenotype_label
Normal      25382
Crackles    15350
Wheezes      6296
Both          360
Name: count, dtype: int64

Counts by original split:
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test               14      1731    5735     1195
train             346     13619   19647     5101

Phase duration summary (seconds):
count    47388.000000
mean         0.940402
std          0.398861
min          0.001000
25%          0.686000
50%          0.887000
75%          1.096000
max          7.139000
Name: duration_seconds, dtype: float64

EXCLUSIONS
reason
rhonchi_or_stridor_overlap    5056
no_respiratory_phase           208
empty_annotation                58
Name: count, dtype: int64

Example candidate windows:
   dataset            recording_id split                                                                                                                         audio_path 

In [15]:

import numpy as np
import pandas as pd

assert not hf_window_preview.empty, "Run the HF window preview cell first."

windows = hf_window_preview.copy()

# Use conservative duration limits for this quality-control pass.
MIN_DURATION = 0.25
MAX_DURATION = 3.00

windows["duration_ok"] = windows["duration_seconds"].between(
    MIN_DURATION, MAX_DURATION, inclusive="both"
)

print("DURATION QUALITY CHECK")
print("Total candidate windows:", len(windows))
print(f"Within {MIN_DURATION:.2f}–{MAX_DURATION:.2f} seconds:",
      int(windows["duration_ok"].sum()))
print("Too short:", int((windows["duration_seconds"] < MIN_DURATION).sum()))
print("Too long:", int((windows["duration_seconds"] > MAX_DURATION).sum()))

print("\nCLASS COUNTS BEFORE DURATION FILTER")
print(windows["phenotype_label"].value_counts())

print("\nCLASS COUNTS AFTER DURATION FILTER")
filtered = windows.loc[windows["duration_ok"]].copy()
print(filtered["phenotype_label"].value_counts())

print("\nCOUNTS BY OFFICIAL SPLIT AFTER DURATION FILTER")
print(pd.crosstab(
    filtered["split"],
    filtered["phenotype_label"]
))

print("\nRECORDING COVERAGE")
for split_name, group in filtered.groupby("split"):
    print(
        split_name,
        "| recordings:", group["recording_id"].nunique(),
        "| windows:", len(group),
        "| patients/dates not inferred yet"
    )

print("\nDURATION SUMMARY BY PHENOTYPE")
print(
    filtered.groupby("phenotype_label")["duration_seconds"]
    .describe()
    .round(3)
)

print("\nQUALITY-control preview only — no files modified.")


DURATION QUALITY CHECK
Total candidate windows: 47388
Within 0.25–3.00 seconds: 47057
Too short: 151
Too long: 180

CLASS COUNTS BEFORE DURATION FILTER
phenotype_label
Normal      25382
Crackles    15350
Wheezes      6296
Both          360
Name: count, dtype: int64

CLASS COUNTS AFTER DURATION FILTER
phenotype_label
Normal      25223
Crackles    15274
Wheezes      6208
Both          352
Name: count, dtype: int64

COUNTS BY OFFICIAL SPLIT AFTER DURATION FILTER
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test               14      1728    5697     1194
train             338     13546   19526     5014

RECORDING COVERAGE
test | recordings: 1850 | windows: 8633 | patients/dates not inferred yet
train | recordings: 7278 | windows: 38424 | patients/dates not inferred yet

DURATION SUMMARY BY PHENOTYPE
                   count   mean    std    min    25%    50%    75%    max
phenotype_label                                                  

In [16]:

import re
import pandas as pd

windows = hf_window_preview.copy()

# Apply the same duration filter used in the previous audit.
windows = windows.loc[
    windows["duration_seconds"].between(0.25, 3.00, inclusive="both")
].copy()

# Extract the date component from both supported filename styles:
# steth_YYYYMMDD_HH_MM_SS
# trunc_YYYY-MM-DD-HH-MM-SS-...
def extract_recording_date(recording_id):
    match = re.match(
        r"^(?:steth_(\d{4})(\d{2})(\d{2})_|"
        r"trunc_(\d{4})-(\d{2})-(\d{2})-)",
        recording_id
    )
    if not match:
        return None

    groups = match.groups()
    if groups[0] is not None:
        year, month, day = groups[0], groups[1], groups[2]
    else:
        year, month, day = groups[3], groups[4], groups[5]

    return f"{year}-{month}-{day}"

windows["recording_date"] = windows["recording_id"].apply(
    extract_recording_date
)

print("DATE EXTRACTION")
print("Windows:", len(windows))
print("Missing date identifiers:", windows["recording_date"].isna().sum())

recording_dates = (
    windows[["recording_id", "recording_date", "split"]]
    .drop_duplicates()
)

date_split_counts = recording_dates.groupby("recording_date")["split"].nunique()
overlap_dates = date_split_counts[date_split_counts > 1]

print("Distinct recordings:", recording_dates["recording_id"].nunique())
print("Distinct extracted dates:", recording_dates["recording_date"].nunique())
print("Dates represented in both official splits:", len(overlap_dates))

if len(overlap_dates):
    print("\nExample dates crossing splits:")
    print(overlap_dates.head(20).to_string())

print("\nDATE-LEVEL CLASS COUNTS")
date_class_counts = (
    windows.groupby(["split", "recording_date", "phenotype_label"])
    .size()
    .unstack(fill_value=0)
)
print(date_class_counts.head(15).to_string())

print("\nRECORDING DATE EXAMPLES")
print(
    recording_dates.head(12).to_string(index=False)
)

print("\nAudit only — no files modified.")


DATE EXTRACTION
Windows: 47057
Missing date identifiers: 0
Distinct recordings: 9128
Distinct extracted dates: 154
Dates represented in both official splits: 0

DATE-LEVEL CLASS COUNTS
phenotype_label       Both  Crackles  Normal  Wheezes
split recording_date                                 
test  2019-08-01         2       146     302        1
      2019-08-02         0         2     162        0
      2019-08-03         0        51      37        0
      2019-08-05         1        27      45        1
      2019-08-06         0        61      26        5
      2019-08-07         0        85     253        3
      2019-08-08         0        41     724      936
      2019-08-09         8       127     362      106
      2019-08-10         1        39     143        1
      2019-08-11         1       186     316       11
      2019-08-12         0        74     135        0
      2019-08-13         0         0      28        0
      2019-08-15         1       138     289        2
     

In [17]:

from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf

HF_OUTPUT_ROOT = (
    PROJECT_ROOT / "data" / "processed" / "unified_phenotype"
    / "hf_lung_v1"
)
HF_AUDIO_OUTPUT = HF_OUTPUT_ROOT / "audio"
HF_AUDIO_OUTPUT.mkdir(parents=True, exist_ok=True)

# Reuse the audited candidate windows and duration filter.
windows_to_process = hf_window_preview.loc[
    hf_window_preview["duration_seconds"].between(
        0.25, 3.00, inclusive="both"
    )
].copy()

# Reuse date extraction already validated in the previous cell.
windows_to_process["recording_date"] = (
    windows_to_process["recording_id"].apply(extract_recording_date)
)

assert windows_to_process["recording_date"].notna().all()
assert not windows_to_process.empty

processed_rows = []
failed_rows = []

for row in windows_to_process.itertuples(index=False):
    source_path = Path(row.audio_path)

    # Keep output paths deterministic and unique per recording/phase.
    output_name = (
        f"{row.recording_id}_phase_{int(row.phase_index):03d}_"
        f"{row.start_time:.3f}_{row.end_time:.3f}.wav"
    )
    output_path = (
        HF_AUDIO_OUTPUT / row.split / output_name
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        audio, sample_rate = sf.read(source_path, dtype="float32")

        # Convert stereo to mono if any file differs from the expected format.
        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        start_sample = int(round(row.start_time * sample_rate))
        end_sample = int(round(row.end_time * sample_rate))

        start_sample = max(0, start_sample)
        end_sample = min(len(audio), end_sample)

        segment = audio[start_sample:end_sample]

        if len(segment) == 0:
            raise ValueError("Empty extracted segment")

        if not np.isfinite(segment).all():
            raise ValueError("Non-finite audio values")

        sf.write(output_path, segment, sample_rate)

        processed_rows.append({
            "dataset": "HF_Lung_V1",
            "recording_id": row.recording_id,
            "recording_date": row.recording_date,
            "split": row.split,
            "source_audio_path": str(source_path),
            "annotation_path": row.annotation_path,
            "phase_index": int(row.phase_index),
            "phase_type": row.phase_type,
            "start_time": float(row.start_time),
            "end_time": float(row.end_time),
            "duration_seconds": len(segment) / sample_rate,
            "sample_rate": int(sample_rate),
            "crackles": int(row.crackles),
            "wheezes": int(row.wheezes),
            "phenotype_label": row.phenotype_label,
            "label_interpretation": (
                "No annotated crackle/wheeze event in phase"
                if row.phenotype_label == "Normal"
                else "Annotation-derived sound phenotype"
            ),
            "overlapping_event_labels": row.overlapping_event_labels,
            "processed_audio_path": str(output_path),
        })

    except Exception as exc:
        failed_rows.append({
            "recording_id": row.recording_id,
            "phase_index": int(row.phase_index),
            "source_audio_path": str(source_path),
            "error": repr(exc),
        })

hf_processed_manifest = pd.DataFrame(processed_rows)
hf_processing_failures = pd.DataFrame(failed_rows)

manifest_path = HF_OUTPUT_ROOT / "hf_phenotype_manifest.csv"
failure_path = HF_OUTPUT_ROOT / "hf_processing_failures.csv"

hf_processed_manifest.to_csv(manifest_path, index=False)
hf_processing_failures.to_csv(failure_path, index=False)

print("HF audio extraction complete")
print("Candidate windows:", len(windows_to_process))
print("Successfully extracted:", len(hf_processed_manifest))
print("Failed:", len(hf_processing_failures))
print("Manifest:", manifest_path)
print("Failure log:", failure_path)

print("\nExtracted class counts:")
print(hf_processed_manifest["phenotype_label"].value_counts())

print("\nExtracted split counts:")
print(pd.crosstab(
    hf_processed_manifest["split"],
    hf_processed_manifest["phenotype_label"]
))

print("\nSample-rate counts:")
print(hf_processed_manifest["sample_rate"].value_counts())

print("\nProcessed files missing from disk:",
      sum(not Path(p).exists()
          for p in hf_processed_manifest["processed_audio_path"]))

print("\nRaw dataset was not modified.")


HF audio extraction complete
Candidate windows: 47057
Successfully extracted: 47057
Failed: 0
Manifest: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/unified_phenotype/hf_lung_v1/hf_phenotype_manifest.csv
Failure log: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/unified_phenotype/hf_lung_v1/hf_processing_failures.csv

Extracted class counts:
phenotype_label
Normal      25223
Crackles    15274
Wheezes      6208
Both          352
Name: count, dtype: int64

Extracted split counts:
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test               14      1728    5697     1194
train             338     13546   19526     5014

Sample-rate counts:
sample_rate
4000    47057
Name: count, dtype: int64

Processed files missing from disk: 0

Raw dataset was not modified.


In [18]:

from pathlib import Path
import pandas as pd

FRAIWAN_ROOT = PROJECT_ROOT / "data" / "raw" / "Fraiwan"
FRAIWAN_AUDIO_DIR = FRAIWAN_ROOT / "Audio Files"

# Locate workbook files without assuming a particular copy or name.
workbooks = sorted(
    p for p in FRAIWAN_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".xlsx", ".xls"}
)

wav_files = sorted(FRAIWAN_AUDIO_DIR.glob("*.wav"))

print("Fraiwan root exists:", FRAIWAN_ROOT.exists())
print("Audio directory exists:", FRAIWAN_AUDIO_DIR.exists())
print("WAV files:", len(wav_files))
print("Workbook files:")
for path in workbooks:
    print(" ", path)

# Inspect workbook sheets and columns.
for workbook in workbooks:
    print("\n" + "=" * 75)
    print("WORKBOOK:", workbook.name)
    print("=" * 75)

    try:
        excel = pd.ExcelFile(workbook)
        print("Sheets:", excel.sheet_names)

        for sheet_name in excel.sheet_names:
            df = pd.read_excel(workbook, sheet_name=sheet_name)
            print(f"\nSheet: {sheet_name}")
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            print(df.head(8).to_string(index=False))
    except Exception as exc:
        print("Workbook inspection error:", repr(exc))

# Inspect filename patterns and prefixes.
print("\nExample WAV filenames:")
for path in wav_files[:20]:
    print(path.name)

print("\nFilename prefix counts:")
prefix_counts = {}
for path in wav_files:
    prefix = path.stem.split("_")[0]
    prefix_counts[prefix] = prefix_counts.get(prefix, 0) + 1

print(prefix_counts)
print("\nInspection only — no files modified.")


Fraiwan root exists: True
Audio directory exists: True
WAV files: 336
Workbook files:
  /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan/Data annotation (1).xlsx

WORKBOOK: Data annotation (1).xlsx
Sheets: ['Sheet1']

Sheet: Sheet1
Shape: (154, 10)
Columns: ['Age', 'Gender', 'Location', 'Sound type', 'Diagnosis', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M']
 Age Gender Location Sound type                      Diagnosis  Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8 Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M
70.0      M    P L L      I E W                         Asthma         NaN         NaN         NaN         NaN      Insipratory: I, Experiatory: E, Wheezes: W, Crackles: C , N:Normal
52.0      F   P L L         E W                         Asthma         NaN         NaN         NaN         NaN                                                       Crep

In [19]:

from pathlib import Path
import pandas as pd
import re

FRAIWAN_ROOT = PROJECT_ROOT / "data" / "raw" / "Fraiwan"
FRAIWAN_AUDIO_DIR = FRAIWAN_ROOT / "Audio Files"
FRAIWAN_WORKBOOK = FRAIWAN_ROOT / "Data annotation (1).xlsx"

audio_files = sorted(FRAIWAN_AUDIO_DIR.glob("*.wav"))
workbook_df = pd.read_excel(FRAIWAN_WORKBOOK)

# Parse filename fields:
# BP101_Asthma,E W,P L M,12,F.wav
def parse_fraiwan_filename(path):
    stem = path.stem
    match = re.match(r"^(BP|DP|EP)(\d+)_(.*)$", stem)
    if not match:
        return None

    prefix, subject_number, remainder = match.groups()
    fields = remainder.split(",")

    if len(fields) < 4:
        return {
            "file_name": path.name,
            "prefix": prefix,
            "subject_id": int(subject_number),
            "filename_diagnosis": None,
            "filename_sound_type": None,
            "filename_location": None,
            "filename_age": None,
            "filename_gender": None,
            "parse_ok": False,
        }

    return {
        "file_name": path.name,
        "prefix": prefix,
        "subject_id": int(subject_number),
        "filename_diagnosis": fields[0].strip(),
        "filename_sound_type": fields[1].strip(),
        "filename_location": fields[2].strip(),
        "filename_age": fields[3].strip(),
        "filename_gender": fields[4].strip() if len(fields) > 4 else None,
        "parse_ok": True,
    }

parsed_rows = [parse_fraiwan_filename(p) for p in audio_files]
filename_df = pd.DataFrame([r for r in parsed_rows if r is not None])

# Normalize workbook values for comparisons.
def norm(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().upper()

for col in ["Sound type", "Diagnosis", "Location", "Gender"]:
    workbook_df[col + "_norm"] = workbook_df[col].apply(norm)

for col in [
    "filename_sound_type", "filename_diagnosis",
    "filename_location", "filename_gender"
]:
    filename_df[col + "_norm"] = filename_df[col].apply(norm)

print("AUDIO FILE PARSING")
print("Audio files:", len(audio_files))
print("Successfully parsed:", int(filename_df["parse_ok"].sum()))
print("Unparsed filenames:", int((~filename_df["parse_ok"]).sum()))
print("Unique numeric subjects:", filename_df["subject_id"].nunique())
print("\nRecordings by prefix:")
print(filename_df["prefix"].value_counts().sort_index())

print("\nWORKBOOK OVERVIEW")
print("Workbook rows:", len(workbook_df))
print("Non-empty sound types:", int((workbook_df["Sound type_norm"] != "").sum()))
print("\nWorkbook sound-type counts:")
print(workbook_df["Sound type_norm"].value_counts(dropna=False).to_string())

print("\nFILENAME SOUND-TYPE COUNTS")
print(filename_df["filename_sound_type_norm"].value_counts(dropna=False).to_string())

# Check filename consistency against workbook values by subject.
# This is a diagnostic comparison, not an assumed one-to-one row join.
wb_subjects = set(range(1, 113))
file_subjects = set(filename_df["subject_id"].unique())

print("\nSUBJECT COVERAGE")
print("Subjects expected (1–112):", len(wb_subjects))
print("Subjects present in filenames:", len(file_subjects))
print("Missing subject IDs:", sorted(wb_subjects - file_subjects))
print("Unexpected subject IDs:", sorted(file_subjects - wb_subjects))

print("\nEXAMPLE PARSED FILENAMES")
print(filename_df.head(15).to_string(index=False))

print("\nInspection only — no files modified.")


AUDIO FILE PARSING
Audio files: 336
Successfully parsed: 336
Unparsed filenames: 0
Unique numeric subjects: 112

Recordings by prefix:
prefix
BP    112
DP    112
EP    112
Name: count, dtype: int64

WORKBOOK OVERVIEW
Workbook rows: 154
Non-empty sound types: 112

Workbook sound-type counts:
Sound type_norm
             42
E W          39
N            35
CREP         23
C             7
I E W         2
I C E W       2
I C B         2
I C           1
BRONCHIAL     1

FILENAME SOUND-TYPE COUNTS
filename_sound_type_norm
E W          117
N            105
CREP          69
C             21
I E W          6
I C E W        6
I C B          6
I C            3
BRONCHIAL      3

SUBJECT COVERAGE
Subjects expected (1–112): 112
Subjects present in filenames: 112
Missing subject IDs: []
Unexpected subject IDs: []

EXAMPLE PARSED FILENAMES
                              file_name prefix  subject_id filename_diagnosis filename_sound_type filename_location filename_age filename_gender  parse_ok filename_s

In [20]:

from pathlib import Path
import pandas as pd
import re

FRAIWAN_ROOT = PROJECT_ROOT / "data" / "raw" / "Fraiwan"
FRAIWAN_AUDIO_DIR = FRAIWAN_ROOT / "Audio Files"
FRAIWAN_WORKBOOK = FRAIWAN_ROOT / "Data annotation (1).xlsx"

wb = pd.read_excel(FRAIWAN_WORKBOOK)

# The populated annotation rows appear to correspond to subjects 1–112.
# Validate this against the filename metadata rather than relying on row order.
wb = wb.copy()
wb["subject_id"] = range(1, len(wb) + 1)

def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

for col in ["Sound type", "Diagnosis", "Location", "Gender"]:
    wb[col] = wb[col].apply(clean_text)

filename_rows = []
for path in sorted(FRAIWAN_AUDIO_DIR.glob("*.wav")):
    match = re.match(r"^(BP|DP|EP)(\d+)_(.*)$", path.stem)
    if not match:
        continue

    prefix, subject_id, remainder = match.groups()
    parts = remainder.split(",")

    filename_rows.append({
        "file_name": path.name,
        "audio_path": str(path),
        "prefix": prefix,
        "subject_id": int(subject_id),
        "filename_diagnosis": clean_text(parts[0]) if len(parts) > 0 else "",
        "filename_sound_type": clean_text(parts[1]) if len(parts) > 1 else "",
        "filename_location": clean_text(parts[2]) if len(parts) > 2 else "",
        "filename_age": clean_text(parts[3]) if len(parts) > 3 else "",
        "filename_gender": clean_text(parts[4]) if len(parts) > 4 else "",
    })

files = pd.DataFrame(filename_rows)

# One workbook annotation row per numeric subject, joined to all three versions.
subject_annotations = wb[
    ["subject_id", "Age", "Gender", "Location", "Sound type", "Diagnosis"]
].copy()

subject_files = files.merge(
    subject_annotations,
    on="subject_id",
    how="left",
    validate="many_to_one",
)

def norm(value):
    return clean_text(value).upper()

for left, right in [
    ("filename_sound_type", "Sound type"),
    ("filename_diagnosis", "Diagnosis"),
    ("filename_location", "Location"),
]:
    subject_files[left + "_match"] = (
        subject_files[left].map(norm) == subject_files[right].map(norm)
    )

print("TOTAL FILES:", len(subject_files))
print("SUBJECTS:", subject_files["subject_id"].nunique())

print("\nMATCH RATE ACROSS BP/DP/EP FILES")
for col in [
    "filename_sound_type_match",
    "filename_diagnosis_match",
    "filename_location_match",
]:
    print(f"{col}: {subject_files[col].sum()}/{len(subject_files)}")

print("\nSOUND TYPE -> CONSERVATIVE LABEL REVIEW")
print(
    subject_annotations["Sound type"]
    .replace("", "<BLANK>")
    .value_counts()
    .to_string()
)

print("\nSUBJECT-LEVEL ANNOTATIONS WITH AMBIGUOUS OR BLANK SOUND TYPES")
ambiguous_codes = {"", "I C B", "I C", "BRONCHIAL"}
review = subject_annotations[
    subject_annotations["Sound type"].str.upper().isin(ambiguous_codes)
]
print(review.to_string(index=False))

print("\nEXAMPLES OF METADATA MISMATCHES")
mismatch_mask = (
    ~subject_files["filename_sound_type_match"]
    | ~subject_files["filename_diagnosis_match"]
    | ~subject_files["filename_location_match"]
)
print(
    subject_files.loc[
        mismatch_mask,
        [
            "file_name", "subject_id", "filename_sound_type", "Sound type",
            "filename_diagnosis", "Diagnosis",
            "filename_location", "Location"
        ],
    ].head(20).to_string(index=False)
)

print("\nInspection only — no files modified.")


TOTAL FILES: 336
SUBJECTS: 112

MATCH RATE ACROSS BP/DP/EP FILES
filename_sound_type_match: 336/336
filename_diagnosis_match: 336/336
filename_location_match: 312/336

SOUND TYPE -> CONSERVATIVE LABEL REVIEW
Sound type
<BLANK>      42
E W          39
N            35
Crep         23
C             7
I E W         2
I C E W       2
I C B         2
I C           1
Bronchial     1

SUBJECT-LEVEL ANNOTATIONS WITH AMBIGUOUS OR BLANK SOUND TYPES
 subject_id  Age Gender Location Sound type                     Diagnosis
          4 72.0      F    P R L        I C Heart Failure + Lung Fibrosis
          6 81.0      M    P R L      I C B              Plueral Effusion
          8 70.0      M    P L L      I C B              Plueral Effusion
         69 64.0      M    P R L  Bronchial                     pneumonia
        113  NaN                                                         
        114  NaN                                                         
        115  NaN                        

In [21]:

from pathlib import Path
import pandas as pd
import re

FRAIWAN_ROOT = PROJECT_ROOT / "data" / "raw" / "Fraiwan"
FRAIWAN_AUDIO_DIR = FRAIWAN_ROOT / "Audio Files"
FRAIWAN_WORKBOOK = FRAIWAN_ROOT / "Data annotation (1).xlsx"

# Only the first 112 rows contain the subject-level annotations.
wb = pd.read_excel(FRAIWAN_WORKBOOK).iloc[:112].copy()
wb["subject_id"] = range(1, 113)

def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

for col in ["Sound type", "Diagnosis", "Location", "Gender"]:
    wb[col] = wb[col].apply(clean_text)

def map_fraiwan_phenotype(sound_type):
    code = clean_text(sound_type).upper()

    if code == "N":
        return "Normal"
    if code in {"C", "CREP"}:
        return "Crackles"
    if code in {"E W", "I E W"}:
        return "Wheezes"
    if code == "I C E W":
        return "Both"

    return None

wb["phenotype_label"] = wb["Sound type"].apply(map_fraiwan_phenotype)

# Parse the actual filenames, retaining original metadata.
file_rows = []
for path in sorted(FRAIWAN_AUDIO_DIR.glob("*.wav")):
    match = re.match(r"^(BP|DP|EP)(\d+)_(.*)$", path.stem)
    if not match:
        continue

    prefix, subject_text, remainder = match.groups()
    parts = remainder.split(",")

    file_rows.append({
        "dataset": "Fraiwan",
        "file_name": path.name,
        "audio_path": str(path),
        "recording_version": prefix,
        "subject_id": int(subject_text),
        "filename_diagnosis": clean_text(parts[0]) if len(parts) > 0 else "",
        "filename_sound_type": clean_text(parts[1]) if len(parts) > 1 else "",
        "filename_location": clean_text(parts[2]) if len(parts) > 2 else "",
        "filename_age": clean_text(parts[3]) if len(parts) > 3 else "",
        "filename_gender": clean_text(parts[4]) if len(parts) > 4 else "",
    })

fraiwan_files = pd.DataFrame(file_rows)

# Join using numeric subject ID, not row order or location.
fraiwan_manifest = fraiwan_files.merge(
    wb[
        [
            "subject_id", "Age", "Gender", "Location",
            "Sound type", "Diagnosis", "phenotype_label"
        ]
    ],
    on="subject_id",
    how="left",
    validate="many_to_one",
)

# Flag label eligibility; don't drop ambiguous recordings silently.
fraiwan_manifest["label_status"] = fraiwan_manifest[
    "phenotype_label"
].apply(lambda x: "eligible" if pd.notna(x) else "excluded_ambiguous_or_missing")

# Subject-level split is assigned later, after we inspect the chosen
# validation strategy. Do not randomly split BP/DP/EP recordings.
fraiwan_manifest["split"] = "unassigned"

print("Total recordings:", len(fraiwan_manifest))
print("Unique subjects:", fraiwan_manifest["subject_id"].nunique())
print("\nRecordings per version:")
print(fraiwan_manifest["recording_version"].value_counts().sort_index())

print("\nSubject-level label counts:")
print(
    wb["phenotype_label"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nRecording-level label counts:")
print(
    fraiwan_manifest["phenotype_label"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nEligibility:")
print(fraiwan_manifest["label_status"].value_counts())

print("\nSubject/version completeness:")
print(
    fraiwan_manifest.groupby("subject_id")["recording_version"]
    .nunique()
    .value_counts()
    .sort_index()
)

print("\nDuplicate audio paths:", fraiwan_manifest["audio_path"].duplicated().sum())
print("Subjects with multiple phenotype labels:",
      fraiwan_manifest.groupby("subject_id")["phenotype_label"]
      .nunique(dropna=True).gt(1).sum())

print("\nExample manifest:")
print(
    fraiwan_manifest[
        [
            "file_name", "subject_id", "recording_version",
            "Sound type", "phenotype_label", "label_status", "split"
        ]
    ].head(15).to_string(index=False)
)

assert len(fraiwan_manifest) == 336
assert fraiwan_manifest["subject_id"].nunique() == 112
assert fraiwan_manifest["audio_path"].is_unique
assert fraiwan_manifest.groupby("subject_id")["phenotype_label"].nunique(
    dropna=True
).le(1).all()

print("\nManifest validated in memory — no files modified.")


Total recordings: 336
Unique subjects: 112

Recordings per version:
recording_version
BP    112
DP    112
EP    112
Name: count, dtype: int64

Subject-level label counts:
phenotype_label
Wheezes     41
Normal      35
Crackles    30
NaN          4
Both         2

Recording-level label counts:
phenotype_label
Wheezes     123
Normal      105
Crackles     90
NaN          12
Both          6

Eligibility:
label_status
eligible                         324
excluded_ambiguous_or_missing     12
Name: count, dtype: int64

Subject/version completeness:
recording_version
3    112
Name: count, dtype: int64

Duplicate audio paths: 0
Subjects with multiple phenotype labels: 0

Example manifest:
                              file_name  subject_id recording_version Sound type phenotype_label label_status      split
               BP100_N,N,P R M,70,F.wav         100                BP          N          Normal     eligible unassigned
        BP101_Asthma,E W,P L M,12,F.wav         101                BP 

In [23]:

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import soundfile as sf

# Use the validated in-memory Fraiwan manifest
assert "fraiwan_manifest" in globals(), (
    "Run the Fraiwan manifest cell first."
)

audio_checks = []

for _, row in fraiwan_manifest.iterrows():
    path = Path(row["audio_path"])

    try:
        info = sf.info(str(path))

        # Hash the file to identify byte-for-byte duplicates
        hasher = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                hasher.update(chunk)

        audio_checks.append({
            "file_name": row["file_name"],
            "subject_id": row["subject_id"],
            "recording_version": row["recording_version"],
            "phenotype_label": row["phenotype_label"],
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "samples": info.frames,
            "duration_seconds": info.duration,
            "file_bytes": path.stat().st_size,
            "sha256": hasher.hexdigest(),
            "read_status": "ok",
        })

    except Exception as exc:
        audio_checks.append({
            "file_name": row["file_name"],
            "subject_id": row["subject_id"],
            "recording_version": row["recording_version"],
            "phenotype_label": row["phenotype_label"],
            "read_status": f"ERROR: {exc}",
        })

fraiwan_audio_audit = pd.DataFrame(audio_checks)

print("========== AUDIO READABILITY ==========")
print(f"Total files checked: {len(fraiwan_audio_audit)}")
print(f"Readable files: {(fraiwan_audio_audit['read_status'] == 'ok').sum()}")
print(f"Failed files: {(fraiwan_audio_audit['read_status'] != 'ok').sum()}")

valid = fraiwan_audio_audit[
    fraiwan_audio_audit["read_status"] == "ok"
].copy()

print("\n========== FORMAT COUNTS ==========")
print("Sample rates:")
print(valid["sample_rate"].value_counts().sort_index())
print("\nChannels:")
print(valid["channels"].value_counts().sort_index())

print("\n========== DURATION BY VERSION ==========")
print(
    valid.groupby("recording_version")["duration_seconds"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(3)
)

print("\n========== DURATION BY PHENOTYPE ==========")
print(
    valid.groupby("phenotype_label", dropna=False)["duration_seconds"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(3)
)

print("\n========== EXACT FILE DUPLICATES ==========")
duplicate_files = valid[
    valid.duplicated("sha256", keep=False)
].sort_values(["sha256", "subject_id", "recording_version"])

if duplicate_files.empty:
    print("No byte-for-byte duplicate WAV files found.")
else:
    print(
        duplicate_files[
            ["file_name", "subject_id", "recording_version", "sha256"]
        ].to_string(index=False)
    )

print("\n========== SUBJECT VERSION DURATION EXAMPLES ==========")
duration_pivot = valid.pivot_table(
    index="subject_id",
    columns="recording_version",
    values="duration_seconds",
    aggfunc="first"
)

print(duration_pivot.head(15).round(3).to_string())

print("\nAudit complete. No files modified.")


========== AUDIO READABILITY ==========
Total files checked: 336
Readable files: 336
Failed files: 0

========== FORMAT COUNTS ==========
Sample rates:
sample_rate
4000    336
Name: count, dtype: int64

Channels:
channels
1    336
Name: count, dtype: int64

========== DURATION BY VERSION ==========
                   count  min  median    mean   max
recording_version                                  
BP                   112  5.0  15.984  17.412  30.0
DP                   112  5.0  15.984  17.412  30.0
EP                   112  5.0  15.920  17.374  30.0

========== DURATION BY PHENOTYPE ==========
                 count     min  median    mean     max
phenotype_label                                       
Both                 6  22.128  24.848  24.848  27.568
Crackles            90   5.000  14.768  15.793  30.000
Normal             105   9.840  17.456  17.815  27.888
Wheezes            123   5.936  15.984  17.122  30.000
NaN                 12  22.064  24.560  24.928  28.528

=========

In [24]:

import hashlib
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path

valid = fraiwan_audio_audit[
    fraiwan_audio_audit["read_status"] == "ok"
].copy()

# Include labels and paths from the original manifest.
audit_details = valid.merge(
    fraiwan_manifest[
        ["file_name", "audio_path", "subject_id",
         "recording_version", "phenotype_label", "Sound type", "Diagnosis"]
    ],
    on=["file_name", "subject_id", "recording_version",
        "phenotype_label"],
    how="left",
    suffixes=("_audit", "_manifest"),
    validate="one_to_one",
)

duplicate_hashes = audit_details[
    audit_details.duplicated("sha256", keep=False)
].sort_values(["sha256", "file_name"])

print("========== DUPLICATE GROUP DETAILS ==========")

for digest, group in duplicate_hashes.groupby("sha256", sort=False):
    print(f"\nSHA-256: {digest}")
    print(group[
        ["file_name", "subject_id", "recording_version",
         "phenotype_label", "Sound type", "Diagnosis",
         "duration_seconds", "file_bytes"]
    ].to_string(index=False))

    paths = group["audio_path"].tolist()
    if len(paths) >= 2:
        first_path = Path(paths[0])
        first_audio, first_sr = sf.read(
            str(first_path), dtype="float32"
        )

        for other_path_text in paths[1:]:
            other_path = Path(other_path_text)
            other_audio, other_sr = sf.read(
                str(other_path), dtype="float32"
            )

            same_samples = (
                first_sr == other_sr
                and first_audio.shape == other_audio.shape
                and np.array_equal(first_audio, other_audio)
            )

            correlation = np.nan
            if (
                first_sr == other_sr
                and first_audio.shape == other_audio.shape
                and first_audio.size > 1
                and np.std(first_audio) > 0
                and np.std(other_audio) > 0
            ):
                correlation = np.corrcoef(
                    first_audio, other_audio
                )[0, 1]

            print(
                f"  Compare {first_path.name} vs {other_path.name}: "
                f"same decoded samples={same_samples}, "
                f"sample rates equal={first_sr == other_sr}, "
                f"correlation={correlation:.6f}"
            )

print("\n========== OVERALL HASH SUMMARY ==========")
print("Unique files:", len(audit_details))
print("Unique SHA-256 hashes:", audit_details["sha256"].nunique())
print("Files involved in duplicate groups:", len(duplicate_hashes))

print("\nNo files modified.")


========== DUPLICATE GROUP DETAILS ==========

SHA-256: 2dbdf57aca466088fc45b4ce53f7f6b5a20042d0349b6b21b527213ae3811dc4
                     file_name  subject_id recording_version phenotype_label Sound type Diagnosis  duration_seconds  file_bytes
BP9_Asthma,E W,P R L ,59,M.wav           9                BP         Wheezes        E W    Asthma            15.344      122796
DP9_Asthma,E W,P R L ,59,M.wav           9                DP         Wheezes        E W    Asthma            15.344      122796
  Compare BP9_Asthma,E W,P R L ,59,M.wav vs DP9_Asthma,E W,P R L ,59,M.wav: same decoded samples=True, sample rates equal=True, correlation=1.000000

SHA-256: 8bd3ed489f89bd6d002bb5e2973bfb61a948df01d40954d6a3bc0c99df67a60f
                      file_name  subject_id recording_version phenotype_label Sound type Diagnosis  duration_seconds  file_bytes
       BP100_N,N,P R M,70,F.wav         100                BP          Normal          N         N            13.552      108460
BP101_Asthma,

In [25]:

import pandas as pd

# Subjects affected by identical audio assigned conflicting labels.
cross_subject_conflict_subjects = {76, 77, 100, 101}

fraiwan_clean_manifest = fraiwan_manifest.copy()

# Default: eligible only when the phenotype mapping is valid.
fraiwan_clean_manifest["modeling_include"] = (
    fraiwan_clean_manifest["phenotype_label"].notna()
)

fraiwan_clean_manifest["quality_status"] = "eligible"
fraiwan_clean_manifest["exclusion_reason"] = ""

# Exclude ambiguous or unmapped labels.
unmapped = fraiwan_clean_manifest["phenotype_label"].isna()

fraiwan_clean_manifest.loc[
    unmapped, "modeling_include"
] = False

fraiwan_clean_manifest.loc[
    unmapped, "quality_status"
] = "excluded_unmapped_label"

fraiwan_clean_manifest.loc[
    unmapped, "exclusion_reason"
] = "Ambiguous or unmapped workbook sound-type annotation"

# Conservatively exclude every recording from affected subjects.
cross_conflict = fraiwan_clean_manifest["subject_id"].isin(
    cross_subject_conflict_subjects
)

fraiwan_clean_manifest.loc[
    cross_conflict, "modeling_include"
] = False

fraiwan_clean_manifest.loc[
    cross_conflict, "quality_status"
] = "excluded_cross_subject_audio_conflict"

fraiwan_clean_manifest.loc[
    cross_conflict, "exclusion_reason"
] = "Exact audio duplicate found across subjects with conflicting labels"

# Subject 9: retain BP, exclude the exact duplicate DP from modeling.
duplicate_subject9 = (
    (fraiwan_clean_manifest["subject_id"] == 9)
    & (fraiwan_clean_manifest["recording_version"] == "DP")
)

fraiwan_clean_manifest.loc[
    duplicate_subject9, "modeling_include"
] = False

fraiwan_clean_manifest.loc[
    duplicate_subject9, "quality_status"
] = "excluded_within_subject_duplicate"

fraiwan_clean_manifest.loc[
    duplicate_subject9, "exclusion_reason"
] = "Exact duplicate of subject 9 BP recording"

# Summaries
print("========== QUALITY-CONTROL SUMMARY ==========")
print("Original recordings:", len(fraiwan_clean_manifest))
print(
    "Modeling-eligible recordings:",
    int(fraiwan_clean_manifest["modeling_include"].sum())
)
print(
    "Excluded recordings:",
    int((~fraiwan_clean_manifest["modeling_include"]).sum())
)

print("\n========== QUALITY STATUS ==========")
print(fraiwan_clean_manifest["quality_status"].value_counts().to_string())

eligible = fraiwan_clean_manifest[
    fraiwan_clean_manifest["modeling_include"]
].copy()

print("\n========== ELIGIBLE PHENOTYPE COUNTS ==========")
print(eligible["phenotype_label"].value_counts().to_string())

print("\n========== ELIGIBLE UNIQUE SUBJECTS ==========")
print(eligible.groupby("phenotype_label")["subject_id"].nunique().to_string())

print("\n========== EXCLUDED SUBJECTS ==========")
print(
    fraiwan_clean_manifest.loc[
        ~fraiwan_clean_manifest["modeling_include"],
        ["subject_id", "recording_version", "phenotype_label",
         "quality_status", "exclusion_reason"]
    ]
    .sort_values(["subject_id", "recording_version"])
    .to_string(index=False)
)

print("\n========== VERSION COUNTS AFTER QC ==========")
print(eligible["recording_version"].value_counts().sort_index().to_string())

print("\nIn-memory quality-control manifest created. No files modified.")


========== QUALITY-CONTROL SUMMARY ==========
Original recordings: 336
Modeling-eligible recordings: 311
Excluded recordings: 25

========== QUALITY STATUS ==========
quality_status
eligible                                 311
excluded_cross_subject_audio_conflict     12
excluded_unmapped_label                   12
excluded_within_subject_duplicate          1

========== ELIGIBLE PHENOTYPE COUNTS ==========
phenotype_label
Wheezes     116
Normal       99
Crackles     90
Both          6

========== ELIGIBLE UNIQUE SUBJECTS ==========
phenotype_label
Both         2
Crackles    30
Normal      33
Wheezes     39

========== EXCLUDED SUBJECTS ==========
 subject_id recording_version phenotype_label                        quality_status                                                    exclusion_reason
          4                BP             NaN               excluded_unmapped_label                Ambiguous or unmapped workbook sound-type annotation
          4                DP           

In [26]:

# Eligible recordings only; raw files remain untouched.
fraiwan_eligible = fraiwan_clean_manifest[
    fraiwan_clean_manifest["modeling_include"]
].copy()

# Every eligible subject must have one consistent phenotype label.
subject_summary = (
    fraiwan_eligible.groupby("subject_id")
    .agg(
        phenotype_label=("phenotype_label", "first"),
        phenotype_count=("phenotype_label", "nunique"),
        eligible_recordings=("file_name", "count"),
        versions=("recording_version", lambda x: ",".join(sorted(x))),
    )
    .reset_index()
)

assert (subject_summary["phenotype_count"] == 1).all(), (
    "A subject has inconsistent phenotype labels."
)

print("========== SUBJECT-LEVEL SUMMARY ==========")
print("Eligible subjects:", len(subject_summary))
print("\nSubjects per phenotype:")
print(
    subject_summary["phenotype_label"]
    .value_counts()
    .reindex(["Normal", "Crackles", "Wheezes", "Both"], fill_value=0)
    .to_string()
)

print("\nEligible recordings per subject:")
print(subject_summary["eligible_recordings"].value_counts().sort_index().to_string())

print("\n========== PROPOSED ROLE ==========")
print("Fraiwan: supplemental training data")
print("Primary evaluation: fixed, subject-disjoint ICBHI test split")
print("Additional evaluation: HF Lung V1 official test split")
print("Both class in Fraiwan: only 2 subjects; insufficient for a reliable independent test")

print("\nNo files modified.")


========== SUBJECT-LEVEL SUMMARY ==========
Eligible subjects: 104

Subjects per phenotype:
phenotype_label
Normal      33
Crackles    30
Wheezes     39
Both         2

Eligible recordings per subject:
eligible_recordings
2      1
3    103

========== PROPOSED ROLE ==========
Fraiwan: supplemental training data
Primary evaluation: fixed, subject-disjoint ICBHI test split
Additional evaluation: HF Lung V1 official test split
Both class in Fraiwan: only 2 subjects; insufficient for a reliable independent test

No files modified.


In [27]:

from sklearn.model_selection import train_test_split
import pandas as pd

# Split subjects, not recordings.
subjects = subject_summary[
    ["subject_id", "phenotype_label"]
].drop_duplicates().copy()

assert subjects["subject_id"].is_unique
assert len(subjects) == 104

# Keep both rare-class subjects in training for the initial supplemental
# training experiment; do not use them to claim independent test performance.
both_subjects = subjects[
    subjects["phenotype_label"] == "Both"
].copy()

other_subjects = subjects[
    subjects["phenotype_label"] != "Both"
].copy()

# Allocate approximately 70% train, 15% validation, 15% internal test
# among the other subjects, stratified by phenotype.
train_val_subjects, test_subjects = train_test_split(
    other_subjects,
    test_size=0.15,
    random_state=42,
    stratify=other_subjects["phenotype_label"],
)

train_subjects, val_subjects = train_test_split(
    train_val_subjects,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_val_subjects["phenotype_label"],
)

train_ids = set(train_subjects["subject_id"]) | set(both_subjects["subject_id"])
val_ids = set(val_subjects["subject_id"])
test_ids = set(test_subjects["subject_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

fraiwan_split_manifest = fraiwan_eligible.copy()
fraiwan_split_manifest["split"] = fraiwan_split_manifest["subject_id"].map(
    lambda sid: (
        "train" if sid in train_ids
        else "validation" if sid in val_ids
        else "test"
    )
)

print("========== SUBJECT SPLIT COUNTS ==========")
print(
    subjects.assign(
        split=subjects["subject_id"].map(
            lambda sid: (
                "train" if sid in train_ids
                else "validation" if sid in val_ids
                else "test"
            )
        )
    ).groupby(["split", "phenotype_label"]).size().unstack(fill_value=0).to_string()
)

print("\n========== RECORDING SPLIT COUNTS ==========")
print(
    fraiwan_split_manifest.groupby(["split", "phenotype_label"])
    .size().unstack(fill_value=0).to_string()
)

print("\n========== SUBJECT LEAKAGE CHECK ==========")
for split_name, ids in [
    ("train", train_ids),
    ("validation", val_ids),
    ("test", test_ids),
]:
    print(f"{split_name}: {len(ids)} unique subjects")

print("\nBoth-class subjects assigned to train:", sorted(both_subjects["subject_id"].tolist()))
print("Manifest created in memory only; no files modified.")


========== SUBJECT SPLIT COUNTS ==========
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test                0         5       5        6
train               2        20      23       27
validation          0         5       5        6

========== RECORDING SPLIT COUNTS ==========
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test                0        15      15       17
train               6        60      69       81
validation          0        15      15       18

========== SUBJECT LEAKAGE CHECK ==========
train: 72 unique subjects
validation: 16 unique subjects
test: 16 unique subjects

Both-class subjects assigned to train: [5, 7]
Manifest created in memory only; no files modified.


In [28]:

import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path

WINDOW_SECONDS = 5.0
MIN_FINAL_WINDOW_SECONDS = 2.5
TARGET_SR = 4000

window_audit_rows = []

for _, row in fraiwan_split_manifest.iterrows():
    path = Path(row["audio_path"])
    info = sf.info(str(path))
    duration = info.duration

    full_windows = int(duration // WINDOW_SECONDS)
    remainder = duration - full_windows * WINDOW_SECONDS

    kept_windows = full_windows
    kept_partial = False

    if remainder >= MIN_FINAL_WINDOW_SECONDS:
        kept_windows += 1
        kept_partial = True

    window_audit_rows.append({
        "subject_id": row["subject_id"],
        "recording_version": row["recording_version"],
        "phenotype_label": row["phenotype_label"],
        "split": row["split"],
        "duration_seconds": duration,
        "full_windows": full_windows,
        "partial_seconds": remainder,
        "partial_kept": kept_partial,
        "total_windows": kept_windows,
        "sample_rate": info.samplerate,
        "channels": info.channels,
    })

fraiwan_window_audit = pd.DataFrame(window_audit_rows)

print("========== WINDOWING AUDIT ==========")
print("Eligible recordings:", len(fraiwan_window_audit))
print("Total proposed windows:", int(fraiwan_window_audit["total_windows"].sum()))
print("Full windows:", int(fraiwan_window_audit["full_windows"].sum()))
print("Kept partial windows:", int(fraiwan_window_audit["partial_kept"].sum()))
print(
    "Discarded short tails:",
    int(
        (
            (fraiwan_window_audit["partial_seconds"] > 0)
            & (~fraiwan_window_audit["partial_kept"])
        ).sum()
    )
)

print("\n========== WINDOWS BY SPLIT AND CLASS ==========")
print(
    fraiwan_window_audit.groupby(["split", "phenotype_label"])["total_windows"]
    .sum().unstack(fill_value=0).to_string()
)

print("\n========== RECORDING DURATION CHECK ==========")
print(
    fraiwan_window_audit.groupby("split")["duration_seconds"]
    .agg(["count", "min", "median", "max"]).round(3).to_string()
)

print("\nAll recordings are 4 kHz mono:",
      bool((fraiwan_window_audit["sample_rate"] == TARGET_SR).all()
           and (fraiwan_window_audit["channels"] == 1).all()))

print("\nDry run only; no audio files written.")


========== WINDOWING AUDIT ==========
Eligible recordings: 311
Total proposed windows: 1086
Full windows: 906
Kept partial windows: 180
Discarded short tails: 113

========== WINDOWS BY SPLIT AND CLASS ==========
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test                0        48      60       63
train              30       210     231      273
validation          0        36      69       66

========== RECORDING DURATION CHECK ==========
            count    min  median   max
split                                 
test           47  9.072  18.672  30.0
train         216  5.000  15.504  30.0
validation     48  5.936  15.920  30.0

All recordings are 4 kHz mono: True

Dry run only; no audio files written.


In [29]:

from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import hashlib
import re

WINDOW_SECONDS = 5.0
MIN_FINAL_WINDOW_SECONDS = 2.5
TARGET_SR = 4000

FRAIWAN_OUT = (
    PROJECT_ROOT / "data" / "processed" /
    "unified_phenotype" / "fraiwan"
)
WINDOW_AUDIO_DIR = FRAIWAN_OUT / "audio"

manifest_rows = []
window_index = 0

for _, row in fraiwan_split_manifest.iterrows():
    source_path = Path(row["audio_path"])
    audio, sr = sf.read(str(source_path), dtype="float32")

    if sr != TARGET_SR or audio.ndim != 1:
        raise ValueError(
            f"Unexpected audio format for {source_path.name}: "
            f"sample_rate={sr}, shape={audio.shape}"
        )

    window_samples = int(WINDOW_SECONDS * TARGET_SR)
    min_partial_samples = int(MIN_FINAL_WINDOW_SECONDS * TARGET_SR)
    total_samples = len(audio)
    full_windows = total_samples // window_samples
    remainder = total_samples % window_samples

    recording_stem = re.sub(r"[^A-Za-z0-9_-]+", "_", source_path.stem)

    for idx in range(full_windows):
        start_sample = idx * window_samples
        end_sample = start_sample + window_samples
        segment = audio[start_sample:end_sample]

        window_index += 1
        out_name = f"{recording_stem}_win{idx:03d}.wav"
        out_path = WINDOW_AUDIO_DIR / str(row["split"]) / out_name
        out_path.parent.mkdir(parents=True, exist_ok=True)

        sf.write(str(out_path), segment, TARGET_SR, subtype="PCM_16")

        manifest_rows.append({
            "dataset": "Fraiwan",
            "subject_id": int(row["subject_id"]),
            "recording_version": row["recording_version"],
            "source_file": source_path.name,
            "source_audio_path": str(source_path),
            "window_index": idx,
            "start_seconds": start_sample / TARGET_SR,
            "end_seconds": end_sample / TARGET_SR,
            "original_window_seconds": WINDOW_SECONDS,
            "stored_window_seconds": WINDOW_SECONDS,
            "was_padded": False,
            "sample_rate": TARGET_SR,
            "phenotype_label": row["phenotype_label"],
            "label_granularity": "recording_level_weak_label",
            "split": row["split"],
            "processed_audio_path": str(out_path.relative_to(PROJECT_ROOT)),
            "quality_status": "eligible",
        })

    # Keep the final partial window only if it is at least 2.5 seconds.
    if remainder >= min_partial_samples:
        start_sample = full_windows * window_samples
        segment = audio[start_sample:]
        original_duration = len(segment) / TARGET_SR

        padded = np.zeros(window_samples, dtype=np.float32)
        padded[:len(segment)] = segment

        idx = full_windows
        window_index += 1
        out_name = f"{recording_stem}_win{idx:03d}.wav"
        out_path = WINDOW_AUDIO_DIR / str(row["split"]) / out_name
        out_path.parent.mkdir(parents=True, exist_ok=True)

        sf.write(str(out_path), padded, TARGET_SR, subtype="PCM_16")

        manifest_rows.append({
            "dataset": "Fraiwan",
            "subject_id": int(row["subject_id"]),
            "recording_version": row["recording_version"],
            "source_file": source_path.name,
            "source_audio_path": str(source_path),
            "window_index": idx,
            "start_seconds": start_sample / TARGET_SR,
            "end_seconds": total_samples / TARGET_SR,
            "original_window_seconds": original_duration,
            "stored_window_seconds": WINDOW_SECONDS,
            "was_padded": True,
            "sample_rate": TARGET_SR,
            "phenotype_label": row["phenotype_label"],
            "label_granularity": "recording_level_weak_label",
            "split": row["split"],
            "processed_audio_path": str(out_path.relative_to(PROJECT_ROOT)),
            "quality_status": "eligible",
        })

fraiwan_window_manifest = pd.DataFrame(manifest_rows)

manifest_path = FRAIWAN_OUT / "fraiwan_window_manifest.csv"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
fraiwan_window_manifest.to_csv(manifest_path, index=False)

print("========== FRAIWAN WINDOW EXPORT ==========")
print("Saved windows:", len(fraiwan_window_manifest))
print("Manifest:", manifest_path)
print("Audio directory:", WINDOW_AUDIO_DIR)
print("\nWindows by split:")
print(fraiwan_window_manifest["split"].value_counts().to_string())
print("\nWindows by class and split:")
print(
    fraiwan_window_manifest.groupby(["split", "phenotype_label"])
    .size().unstack(fill_value=0).to_string()
)
print("\nPadded windows:", int(fraiwan_window_manifest["was_padded"].sum()))
print("All output windows have 5-second stored duration at 4 kHz.")


========== FRAIWAN WINDOW EXPORT ==========
Saved windows: 1086
Manifest: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/unified_phenotype/fraiwan/fraiwan_window_manifest.csv
Audio directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/unified_phenotype/fraiwan/audio

Windows by split:
split
train         744
test          171
validation    171

Windows by class and split:
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test                0        48      60       63
train              30       210     231      273
validation          0        36      69       66

Padded windows: 180
All output windows have 5-second stored duration at 4 kHz.


In [30]:

from pathlib import Path
import pandas as pd
import soundfile as sf

manifest_path = (
    PROJECT_ROOT / "data" / "processed" /
    "unified_phenotype" / "fraiwan" /
    "fraiwan_window_manifest.csv"
)

assert manifest_path.exists(), f"Manifest not found: {manifest_path}"

check_manifest = pd.read_csv(manifest_path)

# Resolve every stored path against the project root.
check_manifest["resolved_path"] = check_manifest[
    "processed_audio_path"
].apply(lambda p: PROJECT_ROOT / p)

missing_paths = check_manifest[
    ~check_manifest["resolved_path"].apply(Path.exists)
]

file_checks = []
for _, row in check_manifest.iterrows():
    path = Path(row["resolved_path"])
    if not path.exists():
        continue

    info = sf.info(str(path))
    file_checks.append({
        "processed_audio_path": row["processed_audio_path"],
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "duration_seconds": info.duration,
        "frames": info.frames,
        "readable": True,
    })

file_checks = pd.DataFrame(file_checks)

print("========== EXPORTED FILE VERIFICATION ==========")
print("Manifest rows:", len(check_manifest))
print("Audio files found:", len(file_checks))
print("Missing audio files:", len(missing_paths))

if len(missing_paths):
    print(missing_paths["processed_audio_path"].head(20).to_string(index=False))

print("\nSample rates:")
print(file_checks["sample_rate"].value_counts().sort_index().to_string())

print("\nChannel counts:")
print(file_checks["channels"].value_counts().sort_index().to_string())

print("\nStored duration counts:")
print(file_checks["duration_seconds"].value_counts().sort_index().to_string())

print("\nAll audio files are 4 kHz mono and 5 seconds:",
      bool(
          len(file_checks) == len(check_manifest)
          and (file_checks["sample_rate"] == 4000).all()
          and (file_checks["channels"] == 1).all()
          and (file_checks["duration_seconds"] == 5.0).all()
      ))

print("\n========== SPLIT AND LABEL COUNTS ==========")
print(check_manifest["split"].value_counts().to_string())
print(
    check_manifest.groupby(["split", "phenotype_label"])
    .size().unstack(fill_value=0).to_string()
)

print("\n========== SUBJECT LEAKAGE ==========")
subject_sets = {
    split: set(group["subject_id"].astype(int))
    for split, group in check_manifest.groupby("split")
}

split_names = list(subject_sets)
for i, split_a in enumerate(split_names):
    for split_b in split_names[i + 1:]:
        overlap = subject_sets[split_a] & subject_sets[split_b]
        print(f"{split_a} vs {split_b}: {len(overlap)} overlapping subjects")
        assert not overlap, f"Subject leakage between {split_a} and {split_b}"

print("\nUnique subjects:", check_manifest["subject_id"].nunique())
print("Padded windows:", int(check_manifest["was_padded"].sum()))
print("Label granularity values:")
print(check_manifest["label_granularity"].value_counts().to_string())

assert len(check_manifest) == 1086
assert len(file_checks) == 1086
assert missing_paths.empty
assert check_manifest["processed_audio_path"].is_unique
assert check_manifest["phenotype_label"].notna().all()
assert (check_manifest["sample_rate"] == 4000).all()
assert (check_manifest["label_granularity"] == "recording_level_weak_label").all()

print("\nPASS: exported files and manifest passed the core checks.")


========== EXPORTED FILE VERIFICATION ==========
Manifest rows: 1086
Audio files found: 1086
Missing audio files: 0

Sample rates:
sample_rate
4000    1086

Channel counts:
channels
1    1086

Stored duration counts:
duration_seconds
5.0    1086

All audio files are 4 kHz mono and 5 seconds: True

========== SPLIT AND LABEL COUNTS ==========
split
train         744
test          171
validation    171
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test                0        48      60       63
train              30       210     231      273
validation          0        36      69       66

========== SUBJECT LEAKAGE ==========
test vs train: 0 overlapping subjects
test vs validation: 0 overlapping subjects
train vs validation: 0 overlapping subjects

Unique subjects: 104
Padded windows: 180
Label granularity values:
label_granularity
recording_level_weak_label    1086

PASS: exported files and manifest passed the core checks.


In [31]:

from pathlib import Path
import pandas as pd

UNIFIED_ROOT = (
    PROJECT_ROOT / "data" / "processed" / "unified_phenotype"
)

manifest_specs = {
    "ICBHI": {
        "path": UNIFIED_ROOT / "icbhi_phenotype_manifest.csv",
        "label_col": "phenotype_label",
        "split_col": "cnn_split",
        "path_col": "processed_path",
    },
    "HF Lung V1": {
        "path": UNIFIED_ROOT / "hf_lung_v1" / "hf_phenotype_manifest.csv",
        "label_col": "phenotype_label",
        "split_col": "split",
        "path_col": "processed_audio_path",
    },
    "Fraiwan": {
        "path": UNIFIED_ROOT / "fraiwan" / "fraiwan_window_manifest.csv",
        "label_col": "phenotype_label",
        "split_col": "split",
        "path_col": "processed_audio_path",
    },
}

for dataset_name, spec in manifest_specs.items():
    print(f"\n========== {dataset_name} ==========")
    manifest_path = spec["path"]

    if not manifest_path.exists():
        print(f"MANIFEST MISSING: {manifest_path}")
        continue

    df = pd.read_csv(manifest_path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    for col in [spec["label_col"], spec["split_col"], spec["path_col"]]:
        print(f"Has {col!r}:", col in df.columns)

    if spec["label_col"] in df.columns:
        print("\nLabel counts:")
        print(df[spec["label_col"]].value_counts(dropna=False).to_string())

    if spec["split_col"] in df.columns:
        print("\nSplit counts:")
        print(df[spec["split_col"]].value_counts(dropna=False).to_string())

    if spec["path_col"] in df.columns:
        paths = df[spec["path_col"]].dropna().astype(str)
        resolved = paths.apply(
            lambda p: Path(p) if Path(p).is_absolute() else PROJECT_ROOT / p
        )
        missing = [str(p) for p in resolved if not p.exists()]
        print("\nMissing referenced audio paths:", len(missing))
        if missing:
            print("Examples:", missing[:5])

    print("\nDuplicate full rows:", int(df.duplicated().sum()))

print("\nManifest audit complete. No files modified.")



========== ICBHI ==========
MANIFEST MISSING: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/unified_phenotype/icbhi_phenotype_manifest.csv

========== HF Lung V1 ==========
Rows: 47057
Columns: ['dataset', 'recording_id', 'recording_date', 'split', 'source_audio_path', 'annotation_path', 'phase_index', 'phase_type', 'start_time', 'end_time', 'duration_seconds', 'sample_rate', 'crackles', 'wheezes', 'phenotype_label', 'label_interpretation', 'overlapping_event_labels', 'processed_audio_path']
Has 'phenotype_label': True
Has 'split': True
Has 'processed_audio_path': True

Label counts:
phenotype_label
Normal      25223
Crackles    15274
Wheezes      6208
Both          352

Split counts:
split
train    38424
test      8633

Missing referenced audio paths: 0

Duplicate full rows: 0

========== Fraiwan ==========
Rows: 1086
Columns: ['dataset', 'subject_id', 'recording_version', 'source_file', 'source_audio_path', 'window_index', 'start_seconds', 'end_seconds', 'origina

In [33]:

from pathlib import Path
import pandas as pd

DATA_ROOT = PROJECT_ROOT / "data" / "processed"
FEATURES_PATH = DATA_ROOT / "features" / "respiratory_cycle_features.csv"
SPLIT_PATH = (
    DATA_ROOT / "deep_learning" / "cnn_baseline" /
    "cnn_patient_disjoint_manifest.csv"
)
OUTPUT_PATH = (
    DATA_ROOT / "unified_phenotype" /
    "icbhi_phenotype_manifest.csv"
)

features = pd.read_csv(FEATURES_PATH)
splits = pd.read_csv(SPLIT_PATH)

# Normalize paths so relative feature paths match absolute split paths.
def normalize_feature_path(value):
    p = Path(str(value))
    if not p.is_absolute():
        p = DATA_ROOT / p
    return str(p.resolve())

def normalize_split_path(value):
    return str(Path(str(value)).resolve())

features["audio_key"] = features["processed_path"].map(
    normalize_feature_path
)
splits["audio_key"] = splits["audio_path"].map(normalize_split_path)

# Check uniqueness before joining.
assert not features["audio_key"].duplicated().any(), (
    "Duplicate paths in the feature table."
)
assert not splits["audio_key"].duplicated().any(), (
    "Duplicate paths in the split manifest."
)

split_cols = [
    "audio_key", "cnn_split", "official_split",
    "patient_id", "recording_id", "diagnosis", "disease_label"
]
split_lookup = splits[split_cols].copy()

# Avoid duplicate patient/recording columns after merging.
features_for_join = features.drop(
    columns=["patient_id", "recording_id", "diagnosis"],
    errors="ignore"
)

icbhi_manifest = features_for_join.merge(
    split_lookup,
    on="audio_key",
    how="left",
    validate="one_to_one",
    indicator=True,
)

unmatched = icbhi_manifest["_merge"] != "both"
print("Feature rows:", len(features))
print("Split rows:", len(splits))
print("Unmatched feature rows:", int(unmatched.sum()))

if unmatched.any():
    print(
        icbhi_manifest.loc[
            unmatched, ["processed_path", "audio_key"]
        ].head(10).to_string(index=False)
    )
    raise ValueError("Path join failed; manifest not saved.")

icbhi_manifest = icbhi_manifest.drop(columns="_merge")

# Use the existing ICBHI cycle-level phenotype labels.
label_map = {
    "normal": "Normal",
    "crackles": "Crackles",
    "wheezes": "Wheezes",
    "both": "Both",
}
icbhi_manifest["phenotype_label"] = (
    icbhi_manifest["sound_label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(label_map)
)

if icbhi_manifest["phenotype_label"].isna().any():
    unknown = icbhi_manifest.loc[
        icbhi_manifest["phenotype_label"].isna(), "sound_label"
    ].value_counts(dropna=False)
    raise ValueError(f"Unmapped sound labels:\n{unknown}")

# Explicitly name the split used for the existing CNN evaluation.
icbhi_manifest["split"] = icbhi_manifest["cnn_split"]
icbhi_manifest["dataset"] = "ICBHI"
icbhi_manifest["label_granularity"] = "respiratory_cycle"
icbhi_manifest["processed_audio_path"] = (
    icbhi_manifest["processed_path"].astype(str)
)

# Verify subject-disjoint splits.
patient_split_counts = icbhi_manifest.groupby("patient_id")["split"].nunique()
assert not (patient_split_counts > 1).any(), (
    "Patient leakage detected across CNN splits."
)

# Verify all referenced audio files exist.
missing_audio = []
for value in icbhi_manifest["processed_audio_path"]:
    p = Path(str(value))
    resolved = p if p.is_absolute() else DATA_ROOT / p
    if not resolved.exists():
        missing_audio.append(str(resolved))

print("\nPhenotype counts:")
print(icbhi_manifest["phenotype_label"].value_counts().to_string())

print("\nCNN split counts:")
print(
    icbhi_manifest.groupby(["split", "phenotype_label"])
    .size().unstack(fill_value=0).to_string()
)

print("\nOfficial source split counts:")
print(icbhi_manifest["official_split"].value_counts(dropna=False).to_string())

print("\nUnique patients:", icbhi_manifest["patient_id"].nunique())
print("Missing audio paths:", len(missing_audio))

if missing_audio:
    print("Examples:", missing_audio[:10])
    raise FileNotFoundError("Some processed audio paths are missing.")

# Save only after validation.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
icbhi_manifest.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)
print("Rows saved:", len(icbhi_manifest))
print("Patient leakage: none")
print("No models retrained or source audio modified.")


Feature rows: 6898
Split rows: 6898
Unmatched feature rows: 0

Phenotype counts:
phenotype_label
Normal      3642
Crackles    1864
Wheezes      886
Both         506

CNN split counts:
phenotype_label  Both  Crackles  Normal  Wheezes
split                                           
test              171       462     705      223
train             316      1219    2342      546
validation         19       183     595      117

Official source split counts:
official_split
train    4142
test     2756

Unique patients: 126
Missing audio paths: 0

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/unified_phenotype/icbhi_phenotype_manifest.csv
Rows saved: 6898
Patient leakage: none
No models retrained or source audio modified.


In [36]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "processed"
UNIFIED_ROOT = DATA_ROOT / "unified_phenotype"

# Rebuild the index from the existing in-memory dataset indexes.
unified_index = pd.concat(
    [icbhi_index, hf_index, fraiwan_index],
    ignore_index=True,
)

def resolve_audio(dataset, value):
    p = Path(str(value)).expanduser()

    if p.is_absolute():
        return p.resolve()

    if dataset == "ICBHI":
        # ICBHI processed_path is relative to data/processed.
        return (DATA_ROOT / p).resolve()

    # HF Lung V1 and Fraiwan paths are project-root-relative.
    return (PROJECT_ROOT / p).resolve()

unified_index["resolved_audio_path"] = [
    resolve_audio(dataset, path)
    for dataset, path in zip(
        unified_index["dataset"], unified_index["audio_path"]
    )
]

missing = unified_index[
    ~unified_index["resolved_audio_path"].apply(Path.exists)
]

print("========== FINAL PATH AUDIT ==========")
print("Total rows:", len(unified_index))
print("Missing audio paths:", len(missing))

print("\nPath check by dataset:")
for dataset, group in unified_index.groupby("dataset"):
    n_missing = int(
        (~group["resolved_audio_path"].apply(Path.exists)).sum()
    )
    print(f"{dataset}: {len(group)} rows, {n_missing} missing")

if len(missing):
    print("\nMissing examples:")
    print(
        missing[["dataset", "audio_path", "resolved_audio_path"]]
        .head(10).to_string(index=False)
    )

duplicate_ids = unified_index.duplicated(
    subset=["dataset", "sample_id"]
)
null_labels = unified_index["phenotype_label"].isna()

print("\nDuplicate dataset/sample IDs:", int(duplicate_ids.sum()))
print("Null phenotype labels:", int(null_labels.sum()))

if missing.empty and not duplicate_ids.any() and not null_labels.any():
    output_path = UNIFIED_ROOT / "unified_phenotype_index.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Save portable paths, not the helper column with absolute paths.
    unified_index.drop(
        columns=["resolved_audio_path"]
    ).to_csv(output_path, index=False)

    print("\nPASS: unified index saved.")
    print("Output:", output_path)
else:
    print("\nIndex not saved; resolve the remaining issues first.")


========== FINAL PATH AUDIT ==========
Total rows: 55041
Missing audio paths: 0

Path check by dataset:
Fraiwan: 1086 rows, 0 missing
HF Lung V1: 47057 rows, 0 missing
ICBHI: 6898 rows, 0 missing

Duplicate dataset/sample IDs: 0
Null phenotype labels: 0

PASS: unified index saved.
Output: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/unified_phenotype/unified_phenotype_index.csv


In [37]:

from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "processed"
UNIFIED_ROOT = DATA_ROOT / "unified_phenotype"

icbhi = pd.read_csv(UNIFIED_ROOT / "icbhi_phenotype_manifest.csv")
hf = pd.read_csv(UNIFIED_ROOT / "hf_lung_v1" / "hf_phenotype_manifest.csv")
fraiwan = pd.read_csv(UNIFIED_ROOT / "fraiwan" / "fraiwan_window_manifest.csv")

# ICBHI cycle durations: inspect the actual saved audio files.
icbhi_durations = []
icbhi_rates = []
icbhi_channels = []
missing_icbhi = []

for rel_path in icbhi["processed_audio_path"].astype(str):
    p = Path(rel_path)
    path = p if p.is_absolute() else DATA_ROOT / p

    if not path.exists():
        missing_icbhi.append(str(path))
        continue

    info = sf.info(str(path))
    icbhi_durations.append(info.duration)
    icbhi_rates.append(info.samplerate)
    icbhi_channels.append(info.channels)

icbhi_duration = pd.Series(icbhi_durations, name="duration_seconds")

print("========== ICBHI AUDIO AUDIT ==========")
print("Manifest rows:", len(icbhi))
print("Audio files inspected:", len(icbhi_duration))
print("Missing files:", len(missing_icbhi))
if len(icbhi_duration):
    print(
        icbhi_duration.describe(
            percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
        ).round(4).to_string()
    )
    print("Sample rates:", pd.Series(icbhi_rates).value_counts().to_dict())
    print("Channels:", pd.Series(icbhi_channels).value_counts().to_dict())

print("\n========== HF LUNG V1 PHASE DURATIONS ==========")
print(
    hf["duration_seconds"]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .round(4).to_string()
)
print("Sample rates:", hf["sample_rate"].value_counts().to_dict())

print("\n========== FRAIWAN STORED WINDOW DURATIONS ==========")
print(
    fraiwan["stored_window_seconds"]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .round(4).to_string()
)
print("Sample rates:", fraiwan["sample_rate"].value_counts().to_dict())
print("Padded windows:", int(fraiwan["was_padded"].sum()))

# Compare the proportion of samples that would need cropping at 3 seconds.
target_seconds = 3.0
print("\n========== PROPOSED 3-SECOND INPUT ==========")
print("Target samples at 4 kHz:", int(target_seconds * 4000))

if len(icbhi_duration):
    print(
        "ICBHI cycles longer than 3 s:",
        int((icbhi_duration > target_seconds).sum()),
        "of", len(icbhi_duration)
    )

print(
    "HF phases longer than 3 s:",
    int((hf["duration_seconds"] > target_seconds).sum()),
    "of", len(hf)
)
print(
    "Fraiwan windows longer than 3 s:",
    int((fraiwan["stored_window_seconds"] > target_seconds).sum()),
    "of", len(fraiwan)
)

if missing_icbhi:
    print("\nMissing ICBHI examples:", missing_icbhi[:10])

print("\nAudit only; no files modified and no models trained.")


========== ICBHI AUDIO AUDIT ==========
Manifest rows: 6898
Audio files inspected: 6898
Missing files: 0
count    6898.0000
mean        2.7006
std         1.1725
min         0.2000
5%          1.0472
25%         1.9322
50%         2.5370
75%         3.3692
95%         4.5910
max        16.1630
Sample rates: {4000: 6898}
Channels: {1: 6898}

========== HF LUNG V1 PHASE DURATIONS ==========
count    47057.0000
mean         0.9332
std          0.3635
min          0.2500
5%           0.4530
25%          0.6880
50%          0.8860
75%          1.0880
95%          1.5920
max          3.0000
Sample rates: {4000: 47057}

========== FRAIWAN STORED WINDOW DURATIONS ==========
count    1086.0
mean        5.0
std         0.0
min         5.0
5%          5.0
25%         5.0
50%         5.0
75%         5.0
95%         5.0
max         5.0
Sample rates: {4000: 1086}
Padded windows: 180

========== PROPOSED 3-SECOND INPUT ==========
Target samples at 4 kHz: 12000
ICBHI cycles longer than 3 s: 2425 of 68

In [38]:

import numpy as np
import pandas as pd

target_seconds = 3.0

duration_audit = pd.DataFrame({
    "duration_seconds": icbhi_duration
})

duration_audit["cropped_seconds"] = (
    duration_audit["duration_seconds"] - target_seconds
).clip(lower=0)

duration_audit["fraction_discarded"] = (
    duration_audit["cropped_seconds"]
    / duration_audit["duration_seconds"].clip(lower=1e-9)
)

print("========== 3-SECOND CROP IMPACT ==========")
print("Total cycles:", len(duration_audit))
print(
    "Cycles requiring cropping:",
    int((duration_audit["duration_seconds"] > target_seconds).sum())
)
print(
    "Cycles requiring padding:",
    int((duration_audit["duration_seconds"] < target_seconds).sum())
)
print(
    "Median discarded duration among cropped cycles:",
    round(
        duration_audit.loc[
            duration_audit["duration_seconds"] > target_seconds,
            "cropped_seconds"
        ].median(),
        3
    ),
    "seconds"
)
print(
    "Mean fraction discarded across all cycles:",
    round(duration_audit["fraction_discarded"].mean() * 100, 2),
    "%"
)
print(
    "95th percentile fraction discarded:",
    round(duration_audit["fraction_discarded"].quantile(0.95) * 100, 2),
    "%"
)

print("\nDuration bands:")
bins = [0, 1, 2, 3, 4, 5, 10, np.inf]
labels = ["<=1s", "1-2s", "2-3s", "3-4s", "4-5s", "5-10s", ">10s"]
duration_audit["duration_band"] = pd.cut(
    duration_audit["duration_seconds"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
print(duration_audit["duration_band"].value_counts().reindex(labels).to_string())

print("\nAudit only; no files modified.")


========== 3-SECOND CROP IMPACT ==========
Total cycles: 6898
Cycles requiring cropping: 2425
Cycles requiring padding: 4468
Median discarded duration among cropped cycles: 0.694 seconds
Mean fraction discarded across all cycles: 7.21 %
95th percentile fraction discarded: 34.65 %

Duration bands:
duration_band
<=1s      302
1-2s     1589
2-3s     2582
3-4s     1634
4-5s      558
5-10s     231
>10s        2

Audit only; no files modified.


In [39]:

from pathlib import Path
import numpy as np
import soundfile as sf

TARGET_SR = 4000
TARGET_SECONDS = 3.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_SECONDS)

def load_standardized_waveform(audio_path, crop_position="center"):
    """
    Load mono 4 kHz audio and return a fixed-length float32 waveform.

    Short signals are zero-padded symmetrically.
    Long signals are cropped to 3 seconds.

    crop_position:
        "center" -> centered crop
        "start"  -> first 3 seconds
    """
    path = Path(audio_path)
    audio, sr = sf.read(str(path), dtype="float32")

    if audio.ndim == 2:
        # Convert multichannel audio to mono if needed.
        audio = audio.mean(axis=1)

    if audio.ndim != 1:
        raise ValueError(f"Unexpected waveform shape {audio.shape}: {path}")

    if sr != TARGET_SR:
        raise ValueError(
            f"Expected {TARGET_SR} Hz, got {sr} Hz: {path}. "
            "Resampling must be explicit."
        )

    if len(audio) == 0:
        raise ValueError(f"Empty audio file: {path}")

    original_samples = len(audio)
    was_cropped = original_samples > TARGET_SAMPLES
    was_padded = original_samples < TARGET_SAMPLES

    if was_cropped:
        if crop_position == "center":
            start = (original_samples - TARGET_SAMPLES) // 2
        elif crop_position == "start":
            start = 0
        else:
            raise ValueError("crop_position must be 'center' or 'start'")

        audio = audio[start:start + TARGET_SAMPLES]

    elif was_padded:
        missing = TARGET_SAMPLES - original_samples
        pad_left = missing // 2
        pad_right = missing - pad_left
        audio = np.pad(audio, (pad_left, pad_right), mode="constant")

    audio = np.asarray(audio, dtype=np.float32)

    if audio.shape != (TARGET_SAMPLES,):
        raise RuntimeError(f"Wrong output shape: {audio.shape}")

    return audio, {
        "original_samples": original_samples,
        "original_duration_seconds": original_samples / TARGET_SR,
        "was_cropped": was_cropped,
        "was_padded": was_padded,
        "output_samples": len(audio),
        "sample_rate": TARGET_SR,
    }

print("Standardized input samples:", TARGET_SAMPLES)
print("Standardized duration:", TARGET_SECONDS, "seconds")
print("Waveform dtype: float32")
print("Function defined. No files modified.")


Standardized input samples: 12000
Standardized duration: 3.0 seconds
Waveform dtype: float32
Function defined. No files modified.


In [40]:

from pathlib import Path
import numpy as np
import pandas as pd

DATA_ROOT = PROJECT_ROOT / "data" / "processed"
UNIFIED_ROOT = DATA_ROOT / "unified_phenotype"

icbhi = pd.read_csv(UNIFIED_ROOT / "icbhi_phenotype_manifest.csv")
hf = pd.read_csv(UNIFIED_ROOT / "hf_lung_v1" / "hf_phenotype_manifest.csv")
fraiwan = pd.read_csv(UNIFIED_ROOT / "fraiwan" / "fraiwan_window_manifest.csv")

# One representative sample from each dataset.
sample_paths = {
    "ICBHI": DATA_ROOT / str(icbhi.iloc[0]["processed_audio_path"]),
    "HF Lung V1": PROJECT_ROOT / str(hf.iloc[0]["processed_audio_path"]),
    "Fraiwan": PROJECT_ROOT / str(fraiwan.iloc[0]["processed_audio_path"]),
}

print("========== STANDARDIZED WAVEFORM TEST ==========")

for dataset_name, path in sample_paths.items():
    waveform, metadata = load_standardized_waveform(path)

    assert waveform.shape == (12000,)
    assert waveform.dtype == np.float32
    assert np.isfinite(waveform).all()
    assert metadata["sample_rate"] == 4000

    print(f"\n{dataset_name}")
    print("File:", path.name)
    print("Output shape:", waveform.shape)
    print("Dtype:", waveform.dtype)
    print("Min / max:", float(waveform.min()), "/", float(waveform.max()))
    print("Original duration:", round(metadata["original_duration_seconds"], 4))
    print("Cropped:", metadata["was_cropped"])
    print("Padded:", metadata["was_padded"])

print("\nPASS: all representative samples standardized correctly.")
print("No files modified and no models trained.")


========== STANDARDIZED WAVEFORM TEST ==========

ICBHI
File: 1b1_Al_001.wav
Output shape: (12000,)
Dtype: float32
Min / max: -0.486724853515625 / 0.999969482421875
Original duration: 0.543
Cropped: False
Padded: True

HF Lung V1
File: steth_20180814_09_37_11_phase_001_1.500_2.457.wav
Output shape: (12000,)
Dtype: float32
Min / max: -0.17138671875 / 0.15478515625
Original duration: 0.957
Cropped: False
Padded: True

Fraiwan
File: BP102_N_N_P_L_L_41_M_win000.wav
Output shape: (12000,)
Dtype: float32
Min / max: -0.53851318359375 / 0.38226318359375
Original duration: 5.0
Cropped: True
Padded: False

PASS: all representative samples standardized correctly.
No files modified and no models trained.


In [41]:

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
UNIFIED_ROOT = PROJECT_ROOT / "data" / "processed" / "unified_phenotype"
INDEX_PATH = UNIFIED_ROOT / "unified_phenotype_index.csv"

unified_index = pd.read_csv(INDEX_PATH)

CLASS_NAMES = ["Normal", "Crackles", "Wheezes", "Both"]
LABEL_TO_ID = {label: i for i, label in enumerate(CLASS_NAMES)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

assert set(unified_index["phenotype_label"].unique()) == set(CLASS_NAMES)

def resolve_index_audio_path(dataset_name, value):
    p = Path(str(value)).expanduser()
    if p.is_absolute():
        return p.resolve()
    if dataset_name == "ICBHI":
        return (PROJECT_ROOT / "data" / "processed" / p).resolve()
    return (PROJECT_ROOT / p).resolve()

class RespiratoryPhenotypeDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True).copy()

        self.df["resolved_audio_path"] = [
            resolve_index_audio_path(ds, path)
            for ds, path in zip(self.df["dataset"], self.df["audio_path"])
        ]

        missing = self.df[
            ~self.df["resolved_audio_path"].apply(Path.exists)
        ]
        if len(missing):
            raise FileNotFoundError(
                f"{len(missing)} audio paths missing; "
                f"examples: {missing['resolved_audio_path'].head(3).tolist()}"
            )

        self.df["label_id"] = self.df["phenotype_label"].map(LABEL_TO_ID)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        waveform, metadata = load_standardized_waveform(
            row["resolved_audio_path"]
        )

        # Shape: (1, 12000), suitable for a 1D CNN.
        waveform = torch.from_numpy(waveform).unsqueeze(0)
        label = torch.tensor(int(row["label_id"]), dtype=torch.long)

        return {
            "waveform": waveform,
            "label": label,
            "dataset": row["dataset"],
            "sample_id": row["sample_id"],
            "subject_id": row["subject_id"],
            "split": row["split"],
            "annotation_granularity": row["annotation_granularity"],
            "label_strength": row["label_strength"],
            "was_cropped": metadata["was_cropped"],
            "was_padded": metadata["was_padded"],
        }

def make_experiment_frame(experiment, split):
    """
    A: ICBHI only
    B: ICBHI + HF Lung V1
    C: ICBHI + HF Lung V1 + Fraiwan

    For training, use only each dataset's designated training partition.
    For validation, use ICBHI's validation partition.
    For primary testing, use ICBHI's held-out test partition.
    """
    if experiment == "A":
        datasets = ["ICBHI"]
    elif experiment == "B":
        datasets = ["ICBHI", "HF Lung V1"]
    elif experiment == "C":
        datasets = ["ICBHI", "HF Lung V1", "Fraiwan"]
    else:
        raise ValueError("experiment must be A, B, or C")

    if split == "train":
        mask = (
            unified_index["dataset"].isin(datasets)
            & (unified_index["split"] == "train")
        )
    elif split == "validation":
        # Identical validation set for fair model selection.
        mask = (
            (unified_index["dataset"] == "ICBHI")
            & (unified_index["split"] == "validation")
        )
    elif split == "test":
        # Identical primary test set for all experiments.
        mask = (
            (unified_index["dataset"] == "ICBHI")
            & (unified_index["split"] == "test")
        )
    else:
        raise ValueError("split must be train, validation, or test")

    result = unified_index.loc[mask].copy()

    if result.empty:
        raise ValueError(f"No samples for experiment {experiment}, split {split}")

    return result

def make_loader(experiment, split, batch_size=32, num_workers=0):
    frame = make_experiment_frame(experiment, split)
    dataset = RespiratoryPhenotypeDataset(frame)

    generator = torch.Generator()
    generator.manual_seed(42)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=num_workers,
        generator=generator if split == "train" else None,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

print("PyTorch version:", torch.__version__)
print("Classes:", LABEL_TO_ID)
print("Unified index rows:", len(unified_index))

for experiment in ["A", "B", "C"]:
    train_frame = make_experiment_frame(experiment, "train")
    print(
        f"Experiment {experiment}: "
        f"{len(train_frame)} training samples; "
        f"datasets={train_frame['dataset'].value_counts().to_dict()}"
    )

print("Dataset and loader definitions ready; no model trained.")


PyTorch version: 2.14.1
Classes: {'Normal': 0, 'Crackles': 1, 'Wheezes': 2, 'Both': 3}
Unified index rows: 55041
Experiment A: 4423 training samples; datasets={'ICBHI': 4423}
Experiment B: 42847 training samples; datasets={'HF Lung V1': 38424, 'ICBHI': 4423}
Experiment C: 43591 training samples; datasets={'HF Lung V1': 38424, 'ICBHI': 4423, 'Fraiwan': 744}
Dataset and loader definitions ready; no model trained.


In [42]:

import torch
from collections import Counter

print("========== DATALOADER BATCH TEST ==========")

for experiment in ["A", "B", "C"]:
    print(f"\n--- Experiment {experiment} ---")

    loader = make_loader(
        experiment=experiment,
        split="train",
        batch_size=8,
        num_workers=0,
    )

    batch = next(iter(loader))

    waveforms = batch["waveform"]
    labels = batch["label"]

    print("Waveform shape:", tuple(waveforms.shape))
    print("Label shape:", tuple(labels.shape))
    print("Waveform dtype:", waveforms.dtype)
    print("Label dtype:", labels.dtype)
    print("Waveforms finite:", bool(torch.isfinite(waveforms).all()))
    print("Label IDs:", labels.tolist())
    print("Dataset sources:", Counter(batch["dataset"]))

    assert waveforms.shape == (8, 1, 12000)
    assert labels.shape == (8,)
    assert waveforms.dtype == torch.float32
    assert labels.dtype == torch.long
    assert torch.isfinite(waveforms).all()
    assert labels.min().item() >= 0
    assert labels.max().item() < len(CLASS_NAMES)

print("\nPASS: all three experiment loaders produced valid batches.")
print("No model trained and no files modified.")


========== DATALOADER BATCH TEST ==========

--- Experiment A ---
Waveform shape: (8, 1, 12000)
Label shape: (8,)
Waveform dtype: torch.float32
Label dtype: torch.int64
Waveforms finite: True
Label IDs: [0, 0, 1, 0, 1, 0, 0, 1]
Dataset sources: Counter({'ICBHI': 8})

--- Experiment B ---
Waveform shape: (8, 1, 12000)
Label shape: (8,)
Waveform dtype: torch.float32
Label dtype: torch.int64
Waveforms finite: True
Label IDs: [0, 0, 1, 1, 1, 1, 1, 0]
Dataset sources: Counter({'HF Lung V1': 7, 'ICBHI': 1})

--- Experiment C ---
Waveform shape: (8, 1, 12000)
Label shape: (8,)
Waveform dtype: torch.float32
Label dtype: torch.int64
Waveforms finite: True
Label IDs: [0, 1, 0, 1, 0, 2, 0, 0]
Dataset sources: Counter({'HF Lung V1': 7, 'ICBHI': 1})

PASS: all three experiment loaders produced valid batches.
No model trained and no files modified.


In [43]:

import pandas as pd

print("========== TRAINING CLASS DISTRIBUTIONS ==========")

for experiment in ["A", "B", "C"]:
    frame = make_experiment_frame(experiment, "train")

    counts = pd.crosstab(
        frame["dataset"],
        frame["phenotype_label"]
    ).reindex(columns=CLASS_NAMES, fill_value=0)

    print(f"\n--- Experiment {experiment} ---")
    print("Counts by dataset and class:")
    print(counts.to_string())

    print("\nWithin-dataset class proportions (%):")
    proportions = counts.div(
        counts.sum(axis=1).replace(0, 1), axis=0
    ) * 100
    print(proportions.round(2).to_string())

    print("\nDataset contribution to each class (%):")
    contribution = counts.div(
        counts.sum(axis=0).replace(0, 1), axis=1
    ) * 100
    print(contribution.round(2).to_string())

print("\n========== FIXED VALIDATION / TEST CHECK ==========")
for split in ["validation", "test"]:
    frame = make_experiment_frame("A", split)
    print(f"{split}: {len(frame)} ICBHI samples")
    print(frame["phenotype_label"].value_counts().reindex(
        CLASS_NAMES, fill_value=0
    ).to_dict())

print("\nAudit only; no model trained and no files modified.")


========== TRAINING CLASS DISTRIBUTIONS ==========

--- Experiment A ---
Counts by dataset and class:
phenotype_label  Normal  Crackles  Wheezes  Both
dataset                                         
ICBHI              2342      1219      546   316

Within-dataset class proportions (%):
phenotype_label  Normal  Crackles  Wheezes  Both
dataset                                         
ICBHI             52.95     27.56    12.34  7.14

Dataset contribution to each class (%):
phenotype_label  Normal  Crackles  Wheezes   Both
dataset                                          
ICBHI             100.0     100.0    100.0  100.0

--- Experiment B ---
Counts by dataset and class:
phenotype_label  Normal  Crackles  Wheezes  Both
dataset                                         
HF Lung V1        19526     13546     5014   338
ICBHI              2342      1219      546   316

Within-dataset class proportions (%):
phenotype_label  Normal  Crackles  Wheezes  Both
dataset                                

In [44]:

import numpy as np
import pandas as pd
from collections import Counter

def add_source_class_sampling_weights(frame):
    """
    Equal probability per dataset, then equal probability per class
    available within that dataset, then uniform sampling within class.
    """
    frame = frame.copy()

    datasets = sorted(frame["dataset"].unique())
    n_datasets = len(datasets)
    weights = np.zeros(len(frame), dtype=np.float64)

    for dataset_name in datasets:
        dataset_mask = frame["dataset"].eq(dataset_name).to_numpy()
        dataset_positions = np.flatnonzero(dataset_mask)
        dataset_frame = frame.iloc[dataset_positions]

        class_counts = dataset_frame["phenotype_label"].value_counts()
        available_classes = list(class_counts.index)
        n_classes = len(available_classes)

        for class_name in available_classes:
            class_positions = dataset_positions[
                dataset_frame["phenotype_label"].to_numpy() == class_name
            ]
            class_count = len(class_positions)

            # Total probability for each dataset is 1 / n_datasets.
            # Classes available within that dataset share it equally.
            weights[class_positions] = (
                1.0 / n_datasets / n_classes / class_count
            )

    weights /= weights.sum()
    frame["sampling_weight"] = weights
    return frame


print("========== SAMPLER WEIGHT AUDIT ==========")

for experiment in ["A", "B", "C"]:
    frame = make_experiment_frame(experiment, "train")
    weighted_frame = add_source_class_sampling_weights(frame)

    print(f"\n--- Experiment {experiment} ---")

    # Expected probability assigned to each dataset.
    dataset_prob = weighted_frame.groupby("dataset")[
        "sampling_weight"
    ].sum()
    print("\nExpected dataset sampling probability:")
    print((dataset_prob * 100).round(2).to_string())

    # Expected probability assigned to each class within each dataset.
    class_prob = (
        weighted_frame.groupby(["dataset", "phenotype_label"])[
            "sampling_weight"
        ].sum().unstack(fill_value=0)
    )
    print("\nExpected class sampling probability within each dataset (%):")
    print((class_prob * 100).round(2).to_string())

    # Overall expected class probability after mixing datasets.
    overall_class_prob = weighted_frame.groupby("phenotype_label")[
        "sampling_weight"
    ].sum().reindex(CLASS_NAMES, fill_value=0)
    print("\nOverall expected class probability (%):")
    print((overall_class_prob * 100).round(2).to_dict())

    assert np.isclose(weighted_frame["sampling_weight"].sum(), 1.0)
    assert np.isfinite(weighted_frame["sampling_weight"]).all()
    assert (weighted_frame["sampling_weight"] > 0).all()

print("\nPASS: weights are finite, positive, and sum to 1.")
print("No model trained and no files modified.")


========== SAMPLER WEIGHT AUDIT ==========

--- Experiment A ---

Expected dataset sampling probability:
dataset
ICBHI    100.0

Expected class sampling probability within each dataset (%):
phenotype_label  Both  Crackles  Normal  Wheezes
dataset                                         
ICBHI            25.0      25.0    25.0     25.0

Overall expected class probability (%):
{'Normal': 25.0, 'Crackles': 25.0, 'Wheezes': 25.0, 'Both': 25.0}

--- Experiment B ---

Expected dataset sampling probability:
dataset
HF Lung V1    50.0
ICBHI         50.0

Expected class sampling probability within each dataset (%):
phenotype_label  Both  Crackles  Normal  Wheezes
dataset                                         
HF Lung V1       12.5      12.5    12.5     12.5
ICBHI            12.5      12.5    12.5     12.5

Overall expected class probability (%):
{'Normal': 25.0, 'Crackles': 25.0, 'Wheezes': 25.0, 'Both': 25.0}

--- Experiment C ---

Expected dataset sampling probability:
dataset
Fraiwan      

In [45]:

import torch
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter

def make_weighted_train_loader(experiment, batch_size=32, seed=42):
    frame = make_experiment_frame(experiment, "train")
    weighted_frame = add_source_class_sampling_weights(frame)

    dataset = RespiratoryPhenotypeDataset(weighted_frame)

    sampler_generator = torch.Generator()
    sampler_generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(
            weighted_frame["sampling_weight"].to_numpy(),
            dtype=torch.double
        ),
        num_samples=len(weighted_frame),
        replacement=True,
        generator=sampler_generator
    )

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        generator=loader_generator
    )

    return loader, weighted_frame


print("========== WEIGHTED SAMPLER TEST ==========")

for experiment in ["A", "B", "C"]:
    loader, weighted_frame = make_weighted_train_loader(
        experiment, batch_size=8, seed=42
    )

    # Audit the sampler indices without decoding every audio file.
    sampled_indices = list(iter(loader.sampler))
    sampled_frame = weighted_frame.iloc[sampled_indices]

    print(f"\n--- Experiment {experiment} ---")
    print("Original training rows:", len(weighted_frame))
    print("Samples drawn per epoch:", len(sampled_indices))

    print("\nSampled dataset counts:")
    print(sampled_frame["dataset"].value_counts().to_dict())

    print("\nSampled class counts:")
    print(
        sampled_frame["phenotype_label"]
        .value_counts()
        .reindex(CLASS_NAMES, fill_value=0)
        .to_dict()
    )

    print("\nSampled class proportions (%):")
    print(
        (
            sampled_frame["phenotype_label"]
            .value_counts(normalize=True)
            .reindex(CLASS_NAMES, fill_value=0) * 100
        ).round(2).to_dict()
    )

    # Test one actual batch from a fresh loader with the same seed.
    batch_loader, _ = make_weighted_train_loader(
        experiment, batch_size=8, seed=42
    )
    batch = next(iter(batch_loader))
    waveforms = batch["waveform"]
    labels = batch["label"]

    print("First batch waveform shape:", tuple(waveforms.shape))
    print("First batch label shape:", tuple(labels.shape))
    print("First batch dataset sources:", Counter(batch["dataset"]))

    assert waveforms.shape == (8, 1, TARGET_SAMPLES)
    assert labels.shape == (8,)
    assert waveforms.dtype == torch.float32
    assert labels.dtype == torch.long
    assert torch.isfinite(waveforms).all()
    assert labels.min().item() >= 0
    assert labels.max().item() < len(CLASS_NAMES)

print("\nPASS: weighted sampler and real audio batches validated.")
print("No model trained and no files modified.")


========== WEIGHTED SAMPLER TEST ==========


/var/folders/xp/3qf723ln63q9ncf3lc_ygq1c0000gp/T/ipykernel_20997/2610927595.py:16: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  weights=torch.as_tensor(



--- Experiment A ---
Original training rows: 4423
Samples drawn per epoch: 4423

Sampled dataset counts:
{'ICBHI': 4423}

Sampled class counts:
{'Normal': 1110, 'Crackles': 1099, 'Wheezes': 1083, 'Both': 1131}

Sampled class proportions (%):
{'Normal': 25.1, 'Crackles': 24.85, 'Wheezes': 24.49, 'Both': 25.57}
First batch waveform shape: (8, 1, 12000)
First batch label shape: (8,)
First batch dataset sources: Counter({'ICBHI': 8})

--- Experiment B ---
Original training rows: 42847
Samples drawn per epoch: 42847

Sampled dataset counts:
{'HF Lung V1': 21474, 'ICBHI': 21373}

Sampled class counts:
{'Normal': 10750, 'Crackles': 10646, 'Wheezes': 10633, 'Both': 10818}

Sampled class proportions (%):
{'Normal': 25.09, 'Crackles': 24.85, 'Wheezes': 24.82, 'Both': 25.25}
First batch waveform shape: (8, 1, 12000)
First batch label shape: (8,)
First batch dataset sources: Counter({'ICBHI': 5, 'HF Lung V1': 3})

--- Experiment C ---
Original training rows: 43591
Samples drawn per epoch: 43591



In [46]:

import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

# Keep the number of sampled examples per epoch comparable across A/B/C.
ICBHI_TRAIN_EPOCH_SIZE = len(
    make_experiment_frame("A", "train")
)

def make_weighted_train_loader(
    experiment,
    batch_size=32,
    seed=42,
    samples_per_epoch=ICBHI_TRAIN_EPOCH_SIZE
):
    frame = make_experiment_frame(experiment, "train")
    weighted_frame = add_source_class_sampling_weights(frame)

    dataset = RespiratoryPhenotypeDataset(weighted_frame)

    # Copy the NumPy array to ensure the tensor uses writable memory.
    weight_array = weighted_frame[
        "sampling_weight"
    ].to_numpy(dtype=np.float64, copy=True)

    sampler_generator = torch.Generator()
    sampler_generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.tensor(weight_array, dtype=torch.double),
        num_samples=samples_per_epoch,
        replacement=True,
        generator=sampler_generator
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False
    ), weighted_frame


print("========== COMPARABLE EPOCH CONFIGURATION ==========")

for experiment in ["A", "B", "C"]:
    loader, frame = make_weighted_train_loader(
        experiment=experiment,
        batch_size=32,
        seed=42
    )

    print(
        f"{experiment}: source rows={len(frame):,}, "
        f"sampled examples/epoch={len(loader.sampler):,}, "
        f"batches/epoch={len(loader):,}"
    )

    assert len(loader.sampler) == ICBHI_TRAIN_EPOCH_SIZE

print("PASS: all experiments use the same sampled epoch size.")
print("No model trained and no files modified.")


========== COMPARABLE EPOCH CONFIGURATION ==========
A: source rows=4,423, sampled examples/epoch=4,423, batches/epoch=139
B: source rows=42,847, sampled examples/epoch=4,423, batches/epoch=139
C: source rows=43,591, sampled examples/epoch=4,423, batches/epoch=139
PASS: all experiments use the same sampled epoch size.
No model trained and no files modified.


In [47]:

import torch
import numpy as np

print("========== CNN ENVIRONMENT CHECK ==========")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", hasattr(torch.backends, "mps")
      and torch.backends.mps.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Selected device:", DEVICE)
print("Input shape per sample:", (1, TARGET_SAMPLES))
print("Sampling rate:", TARGET_SR, "Hz")
print("Input duration:", TARGET_SECONDS, "seconds")
print("Number of classes:", len(CLASS_NAMES))
print("Class mapping:", LABEL_TO_ID)

assert TARGET_SAMPLES == TARGET_SR * int(TARGET_SECONDS)
assert CLASS_NAMES == ["Normal", "Crackles", "Wheezes", "Both"]

print("PASS: environment and input configuration validated.")


========== CNN ENVIRONMENT CHECK ==========
PyTorch version: 2.14.1
CUDA available: False
MPS available: True
Selected device: mps
Input shape per sample: (1, 12000)
Sampling rate: 4000 Hz
Input duration: 3.0 seconds
Number of classes: 4
Class mapping: {'Normal': 0, 'Crackles': 1, 'Wheezes': 2, 'Both': 3}
PASS: environment and input configuration validated.


In [48]:

import torch
import torch.nn as nn

class CompactRespiratoryCNN(nn.Module):
    def __init__(self, num_classes=4, dropout=0.25):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=4),

            nn.Conv1d(16, 32, kernel_size=9, stride=1, padding=4, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=4),

            nn.Conv1d(32, 64, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=4),

            nn.Conv1d(64, 64, kernel_size=5, stride=1, padding=2, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


model = CompactRespiratoryCNN(num_classes=len(CLASS_NAMES)).to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("========== CNN MODEL AUDIT ==========")
print("Device:", DEVICE)
print("Model:", model.__class__.__name__)
print("Total parameters:", f"{parameter_count:,}")
print("Trainable parameters:", f"{trainable_count:,}")

# Verify output shape with a synthetic batch.
dummy_input = torch.zeros(2, 1, TARGET_SAMPLES, device=DEVICE)

model.eval()
with torch.no_grad():
    dummy_output = model(dummy_input)

print("Input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(dummy_output.shape))

assert dummy_output.shape == (2, len(CLASS_NAMES))
assert torch.isfinite(dummy_output).all()

print("PASS: model forward pass validated.")
print("No model trained and no files modified.")


========== CNN MODEL AUDIT ==========
Device: mps
Model: CompactRespiratoryCNN
Total parameters: 40,276
Trainable parameters: 40,276
Input shape: (2, 1, 12000)
Output shape: (2, 4)
PASS: model forward pass validated.
No model trained and no files modified.


In [49]:

import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from pathlib import Path

SEED = 42
BATCH_SIZE = 32
MAX_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 7
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

CHECKPOINT_DIR = (
    PROJECT_ROOT / "models" / "phenotype_cnn"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def set_reproducible_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch, "mps") and hasattr(torch.mps, "manual_seed"):
        try:
            torch.mps.manual_seed(seed)
        except Exception:
            pass

def calculate_class_weights(frame):
    """
    Inverse-frequency class weights normalized to mean 1.
    Calculate using training data only.
    """
    counts = (
        frame["phenotype_label"]
        .value_counts()
        .reindex(CLASS_NAMES, fill_value=0)
        .astype(float)
    )

    if (counts == 0).any():
        raise ValueError(
            f"Missing training classes for class-weighted loss: "
            f"{counts[counts == 0].index.tolist()}"
        )

    weights = 1.0 / counts
    weights = weights / weights.mean()

    return torch.tensor(
        weights.to_numpy(),
        dtype=torch.float32,
        device=DEVICE
    )

def evaluate_predictions(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred,
            labels=list(range(len(CLASS_NAMES))),
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            y_true, y_pred,
            labels=list(range(len(CLASS_NAMES)))
        ),
        "classification_report": classification_report(
            y_true, y_pred,
            labels=list(range(len(CLASS_NAMES))),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )
    }

set_reproducible_seed()

# Sanity-check the metric function with synthetic labels.
y_true_demo = np.array([0, 0, 1, 1, 2, 2, 3, 3])
y_pred_demo = np.array([0, 1, 1, 1, 2, 0, 3, 2])
demo_metrics = evaluate_predictions(y_true_demo, y_pred_demo)

print("========== TRAINING CONFIGURATION ==========")
print("Device:", DEVICE)
print("Seed:", SEED)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Early-stopping patience:", EARLY_STOPPING_PATIENCE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Metric sanity check - macro-F1:", round(demo_metrics["macro_f1"], 4))
print("Metric sanity check - balanced accuracy:",
      round(demo_metrics["balanced_accuracy"], 4))

assert 0 <= demo_metrics["macro_f1"] <= 1
assert demo_metrics["confusion_matrix"].shape == (4, 4)

print("PASS: training configuration and metrics validated.")
print("No model trained.")


========== TRAINING CONFIGURATION ==========
Device: mps
Seed: 42
Batch size: 32
Maximum epochs: 40
Early-stopping patience: 7
Learning rate: 0.001
Weight decay: 0.0001
Checkpoint directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/models/phenotype_cnn
Metric sanity check - macro-F1: 0.6167
Metric sanity check - balanced accuracy: 0.625
PASS: training configuration and metrics validated.
No model trained.


In [50]:

import copy
import time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

def make_natural_train_loader(experiment, batch_size=BATCH_SIZE, seed=SEED):
    frame = make_experiment_frame(experiment, "train")
    dataset = RespiratoryPhenotypeDataset(frame)

    generator = torch.Generator()
    generator.manual_seed(seed)

    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        generator=generator
    )
    return loader, frame


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_examples = 0
    all_true = []
    all_pred = []

    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for batch in loader:
            waveforms = batch["waveform"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(waveforms)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            batch_size_actual = labels.size(0)
            total_loss += loss.item() * batch_size_actual
            total_examples += batch_size_actual

            predictions = logits.argmax(dim=1)
            all_true.extend(labels.detach().cpu().numpy().tolist())
            all_pred.extend(predictions.detach().cpu().numpy().tolist())

    average_loss = total_loss / max(total_examples, 1)
    macro_f1 = f1_score(
        all_true,
        all_pred,
        labels=list(range(len(CLASS_NAMES))),
        average="macro",
        zero_division=0
    )

    return {
        "loss": average_loss,
        "macro_f1": macro_f1,
        "y_true": np.asarray(all_true),
        "y_pred": np.asarray(all_pred)
    }


def train_model(
    experiment,
    strategy="natural",
    seed=SEED,
    max_epochs=MAX_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE
):
    """
    strategy:
      natural   - shuffled natural training distribution
      balanced  - dataset-aware, class-balanced sampler
      weighted  - natural sampling with inverse-frequency class-weighted loss

    Selection always uses the fixed ICBHI validation split.
    """
    set_reproducible_seed(seed)

    if strategy == "natural" or strategy == "weighted":
        train_loader, train_frame = make_natural_train_loader(
            experiment, batch_size=BATCH_SIZE, seed=seed
        )
    elif strategy == "balanced":
        train_loader, train_frame = make_weighted_train_loader(
            experiment,
            batch_size=BATCH_SIZE,
            seed=seed,
            samples_per_epoch=ICBHI_TRAIN_EPOCH_SIZE
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    validation_loader = make_loader(
        experiment, "validation", batch_size=BATCH_SIZE, num_workers=0
    )

    model = CompactRespiratoryCNN(
        num_classes=len(CLASS_NAMES)
    ).to(DEVICE)

    if strategy == "weighted":
        class_weights = calculate_class_weights(train_frame)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    checkpoint_path = CHECKPOINT_DIR / (
        f"cnn_{experiment}_{strategy}_seed{seed}.pt"
    )

    best_f1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    print(
        f"Starting experiment={experiment}, strategy={strategy}, "
        f"train rows={len(train_frame):,}, "
        f"sampled epoch size={len(train_loader.sampler) if hasattr(train_loader, 'sampler') else len(train_loader.dataset):,}"
    )

    for epoch in range(1, max_epochs + 1):
        start_time = time.time()

        train_result = run_epoch(
            model, train_loader, criterion, optimizer=optimizer
        )
        val_result = run_epoch(
            model, validation_loader, criterion, optimizer=None
        )

        record = {
            "epoch": epoch,
            "train_loss": train_result["loss"],
            "train_macro_f1": train_result["macro_f1"],
            "val_loss": val_result["loss"],
            "val_macro_f1": val_result["macro_f1"],
            "seconds": time.time() - start_time
        }
        history.append(record)

        print(
            f"Epoch {epoch:02d}/{max_epochs} | "
            f"train loss={record['train_loss']:.4f}, "
            f"train macro-F1={record['train_macro_f1']:.4f} | "
            f"val loss={record['val_loss']:.4f}, "
            f"val macro-F1={record['val_macro_f1']:.4f} | "
            f"{record['seconds']:.1f}s"
        )

        if val_result["macro_f1"] > best_f1:
            best_f1 = val_result["macro_f1"]
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "experiment": experiment,
                "strategy": strategy,
                "seed": seed,
                "epoch": epoch,
                "best_val_macro_f1": best_f1,
                "class_names": CLASS_NAMES,
                "target_sr": TARGET_SR,
                "target_samples": TARGET_SAMPLES
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    print(
        f"Finished: best epoch={best_epoch}, "
        f"best validation macro-F1={best_f1:.4f}"
    )
    print("Checkpoint:", checkpoint_path)

    return {
        "experiment": experiment,
        "strategy": strategy,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_f1,
        "history": history,
        "checkpoint_path": str(checkpoint_path)
    }


print("PASS: training functions defined.")
print("No training started and no checkpoint saved.")


PASS: training functions defined.
No training started and no checkpoint saved.


In [51]:

baseline_A_natural = train_model(
    experiment="A",
    strategy="natural",
    seed=42,
    max_epochs=40,
    patience=7
)

print("\n========== EXPERIMENT A BASELINE SUMMARY ==========")
print("Best epoch:", baseline_A_natural["best_epoch"])
print(
    "Best validation macro-F1:",
    round(baseline_A_natural["best_val_macro_f1"], 4)
)
print("Checkpoint:", baseline_A_natural["checkpoint_path"])


Starting experiment=A, strategy=natural, train rows=4,423, sampled epoch size=4,423
Epoch 01/40 | train loss=1.1327, train macro-F1=0.2721 | val loss=0.9321, val macro-F1=0.2054 | 9.6s
Epoch 02/40 | train loss=1.0588, train macro-F1=0.2764 | val loss=0.9556, val macro-F1=0.2935 | 3.7s
Epoch 03/40 | train loss=1.0448, train macro-F1=0.2823 | val loss=0.9519, val macro-F1=0.2954 | 3.6s
Epoch 04/40 | train loss=1.0327, train macro-F1=0.2965 | val loss=0.9501, val macro-F1=0.2865 | 3.6s
Epoch 05/40 | train loss=1.0281, train macro-F1=0.2958 | val loss=0.9734, val macro-F1=0.2695 | 3.7s
Epoch 06/40 | train loss=1.0168, train macro-F1=0.3173 | val loss=0.9319, val macro-F1=0.2239 | 3.7s
Epoch 07/40 | train loss=1.0113, train macro-F1=0.3206 | val loss=1.0374, val macro-F1=0.3057 | 3.6s
Epoch 08/40 | train loss=1.0056, train macro-F1=0.3240 | val loss=0.9050, val macro-F1=0.2940 | 3.6s
Epoch 09/40 | train loss=0.9911, train macro-F1=0.3425 | val loss=1.0246, val macro-F1=0.2480 | 3.6s
Epoch 1

In [52]:

import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

checkpoint_path = (
    CHECKPOINT_DIR / "cnn_A_natural_seed42.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

best_model_A = CompactRespiratoryCNN(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

best_model_A.load_state_dict(checkpoint["model_state_dict"])
best_model_A.eval()

validation_loader = make_loader(
    "A", "validation", batch_size=BATCH_SIZE, num_workers=0
)

y_true, y_pred = [], []

with torch.no_grad():
    for batch in validation_loader:
        waveforms = batch["waveform"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        logits = best_model_A(waveforms)
        predictions = logits.argmax(dim=1)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(predictions.cpu().numpy().tolist())

print("========== EXPERIMENT A VALIDATION REPORT ==========")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Saved best validation macro-F1:",
      round(checkpoint["best_val_macro_f1"], 4))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print("Class order:", CLASS_NAMES)
print(confusion_matrix(
    y_true, y_pred,
    labels=list(range(len(CLASS_NAMES)))
))

print("\nValidation-only evaluation; test set untouched.")


========== EXPERIMENT A VALIDATION REPORT ==========
Checkpoint epoch: 12
Saved best validation macro-F1: 0.3291

Classification report:
              precision    recall  f1-score   support

      Normal     0.7145    0.8454    0.7744       595
    Crackles     0.3621    0.3443    0.3529       183
     Wheezes     0.4516    0.1197    0.1892       117
        Both     0.0000    0.0000    0.0000        19

    accuracy                         0.6346       914
   macro avg     0.3820    0.3273    0.3291       914
weighted avg     0.5954    0.6346    0.5990       914

Confusion matrix (rows=true, columns=predicted):
Class order: ['Normal', 'Crackles', 'Wheezes', 'Both']
[[503  81  11   0]
 [113  63   5   2]
 [ 79  21  14   3]
 [  9   9   1   0]]

Validation-only evaluation; test set untouched.


In [53]:

baseline_A_weighted = train_model(
    experiment="A",
    strategy="weighted",
    seed=42,
    max_epochs=40,
    patience=7
)

print("\n========== EXPERIMENT A WEIGHTED-LOSS SUMMARY ==========")
print("Best epoch:", baseline_A_weighted["best_epoch"])
print(
    "Best validation macro-F1:",
    round(baseline_A_weighted["best_val_macro_f1"], 4)
)
print("Checkpoint:", baseline_A_weighted["checkpoint_path"])


Starting experiment=A, strategy=weighted, train rows=4,423, sampled epoch size=4,423
Epoch 01/40 | train loss=1.3465, train macro-F1=0.3166 | val loss=1.3346, val macro-F1=0.2684 | 5.9s
Epoch 02/40 | train loss=1.3049, train macro-F1=0.3443 | val loss=1.2429, val macro-F1=0.2881 | 3.7s
Epoch 03/40 | train loss=1.2802, train macro-F1=0.3605 | val loss=1.3368, val macro-F1=0.3030 | 3.7s
Epoch 04/40 | train loss=1.2728, train macro-F1=0.3701 | val loss=1.3530, val macro-F1=0.2878 | 3.7s
Epoch 05/40 | train loss=1.2645, train macro-F1=0.3831 | val loss=1.2902, val macro-F1=0.2701 | 3.7s
Epoch 06/40 | train loss=1.2453, train macro-F1=0.3819 | val loss=1.2391, val macro-F1=0.3009 | 3.7s
Epoch 07/40 | train loss=1.2352, train macro-F1=0.3814 | val loss=1.2391, val macro-F1=0.3019 | 3.6s
Epoch 08/40 | train loss=1.2283, train macro-F1=0.3966 | val loss=1.2413, val macro-F1=0.3145 | 3.6s
Epoch 09/40 | train loss=1.2081, train macro-F1=0.4024 | val loss=1.4704, val macro-F1=0.2153 | 3.7s
Epoch 

In [54]:

import torch
from sklearn.metrics import classification_report, confusion_matrix

checkpoint_path = CHECKPOINT_DIR / "cnn_A_weighted_seed42.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

weighted_model_A = CompactRespiratoryCNN(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

weighted_model_A.load_state_dict(checkpoint["model_state_dict"])
weighted_model_A.eval()

validation_loader = make_loader(
    "A", "validation", batch_size=BATCH_SIZE, num_workers=0
)

y_true_weighted, y_pred_weighted = [], []

with torch.no_grad():
    for batch in validation_loader:
        waveforms = batch["waveform"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        logits = weighted_model_A(waveforms)
        predictions = logits.argmax(dim=1)

        y_true_weighted.extend(labels.cpu().numpy().tolist())
        y_pred_weighted.extend(predictions.cpu().numpy().tolist())

print("========== EXPERIMENT A WEIGHTED VALIDATION REPORT ==========")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Saved best validation macro-F1:",
      round(checkpoint["best_val_macro_f1"], 4))

print("\nClassification report:")
print(classification_report(
    y_true_weighted,
    y_pred_weighted,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print("Class order:", CLASS_NAMES)
print(confusion_matrix(
    y_true_weighted,
    y_pred_weighted,
    labels=list(range(len(CLASS_NAMES)))
))

print("\nValidation-only evaluation; test set untouched.")


========== EXPERIMENT A WEIGHTED VALIDATION REPORT ==========
Checkpoint epoch: 11
Saved best validation macro-F1: 0.3484

Classification report:
              precision    recall  f1-score   support

      Normal     0.7345    0.6975    0.7155       595
    Crackles     0.3750    0.1967    0.2581       183
     Wheezes     0.2882    0.4188    0.3415       117
        Both     0.0482    0.2105    0.0784        19

    accuracy                         0.5514       914
   macro avg     0.3615    0.3809    0.3484       914
weighted avg     0.5911    0.5514    0.5628       914

Confusion matrix (rows=true, columns=predicted):
Class order: ['Normal', 'Crackles', 'Wheezes', 'Both']
[[415  55 101  24]
 [ 93  36  19  35]
 [ 44   4  49  20]
 [ 13   1   1   4]]

Validation-only evaluation; test set untouched.


In [55]:

baseline_A_balanced = train_model(
    experiment="A",
    strategy="balanced",
    seed=42,
    max_epochs=40,
    patience=7
)

print("\n========== EXPERIMENT A BALANCED-SAMPLER SUMMARY ==========")
print("Best epoch:", baseline_A_balanced["best_epoch"])
print(
    "Best validation macro-F1:",
    round(baseline_A_balanced["best_val_macro_f1"], 4)
)
print("Checkpoint:", baseline_A_balanced["checkpoint_path"])


Starting experiment=A, strategy=balanced, train rows=4,423, sampled epoch size=4,423
Epoch 01/40 | train loss=1.3257, train macro-F1=0.3495 | val loss=1.1915, val macro-F1=0.3161 | 5.3s
Epoch 02/40 | train loss=1.3015, train macro-F1=0.3743 | val loss=1.1956, val macro-F1=0.2786 | 4.2s
Epoch 03/40 | train loss=1.2521, train macro-F1=0.4090 | val loss=1.2333, val macro-F1=0.3227 | 3.9s
Epoch 04/40 | train loss=1.2305, train macro-F1=0.4243 | val loss=1.4107, val macro-F1=0.2610 | 3.8s
Epoch 05/40 | train loss=1.1985, train macro-F1=0.4448 | val loss=1.2357, val macro-F1=0.2953 | 3.9s
Epoch 06/40 | train loss=1.1886, train macro-F1=0.4588 | val loss=1.3068, val macro-F1=0.3000 | 3.8s
Epoch 07/40 | train loss=1.1720, train macro-F1=0.4721 | val loss=1.1629, val macro-F1=0.3593 | 3.9s
Epoch 08/40 | train loss=1.1310, train macro-F1=0.5019 | val loss=1.2677, val macro-F1=0.2990 | 3.8s
Epoch 09/40 | train loss=1.1125, train macro-F1=0.5232 | val loss=1.3760, val macro-F1=0.2750 | 3.9s
Epoch 

In [56]:

import torch
from sklearn.metrics import classification_report, confusion_matrix

checkpoint_path = CHECKPOINT_DIR / "cnn_A_balanced_seed42.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

balanced_model_A = CompactRespiratoryCNN(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

balanced_model_A.load_state_dict(checkpoint["model_state_dict"])
balanced_model_A.eval()

validation_loader = make_loader(
    "A", "validation", batch_size=BATCH_SIZE, num_workers=0
)

y_true_balanced, y_pred_balanced = [], []

with torch.no_grad():
    for batch in validation_loader:
        waveforms = batch["waveform"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        logits = balanced_model_A(waveforms)
        predictions = logits.argmax(dim=1)

        y_true_balanced.extend(labels.cpu().numpy().tolist())
        y_pred_balanced.extend(predictions.cpu().numpy().tolist())

print("========== EXPERIMENT A BALANCED VALIDATION REPORT ==========")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Saved best validation macro-F1:",
      round(checkpoint["best_val_macro_f1"], 4))

print("\nClassification report:")
print(classification_report(
    y_true_balanced,
    y_pred_balanced,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print("Class order:", CLASS_NAMES)
print(confusion_matrix(
    y_true_balanced,
    y_pred_balanced,
    labels=list(range(len(CLASS_NAMES)))
))

print("\nValidation-only evaluation; test set untouched.")


========== EXPERIMENT A BALANCED VALIDATION REPORT ==========
Checkpoint epoch: 12
Saved best validation macro-F1: 0.4106

Classification report:
              precision    recall  f1-score   support

      Normal     0.7583    0.6538    0.7022       595
    Crackles     0.4121    0.4098    0.4110       183
     Wheezes     0.3589    0.6410    0.4601       117
        Both     0.1000    0.0526    0.0690        19

    accuracy                         0.5908       914
   macro avg     0.4073    0.4393    0.4106       914
weighted avg     0.6242    0.5908    0.5997       914

Confusion matrix (rows=true, columns=predicted):
Class order: ['Normal', 'Crackles', 'Wheezes', 'Both']
[[389  90 112   4]
 [ 86  75  18   4]
 [ 28  13  75   1]
 [ 10   4   4   1]]

Validation-only evaluation; test set untouched.


In [57]:

baseline_B_balanced = train_model(
    experiment="B",
    strategy="balanced",
    seed=42,
    max_epochs=40,
    patience=7
)

print("\n========== EXPERIMENT B BALANCED-SAMPLER SUMMARY ==========")
print("Best epoch:", baseline_B_balanced["best_epoch"])
print(
    "Best validation macro-F1:",
    round(baseline_B_balanced["best_val_macro_f1"], 4)
)
print("Checkpoint:", baseline_B_balanced["checkpoint_path"])


Starting experiment=B, strategy=balanced, train rows=42,847, sampled epoch size=4,423
Epoch 01/40 | train loss=1.3717, train macro-F1=0.3164 | val loss=1.3576, val macro-F1=0.2449 | 5.3s
Epoch 02/40 | train loss=1.3451, train macro-F1=0.3364 | val loss=1.2227, val macro-F1=0.4082 | 4.6s
Epoch 03/40 | train loss=1.3383, train macro-F1=0.3601 | val loss=1.3353, val macro-F1=0.2639 | 4.3s
Epoch 04/40 | train loss=1.3131, train macro-F1=0.3787 | val loss=1.3204, val macro-F1=0.2942 | 4.6s
Epoch 05/40 | train loss=1.3028, train macro-F1=0.3871 | val loss=1.2562, val macro-F1=0.2688 | 4.6s
Epoch 06/40 | train loss=1.2951, train macro-F1=0.3936 | val loss=1.2630, val macro-F1=0.3381 | 4.5s
Epoch 07/40 | train loss=1.2732, train macro-F1=0.4140 | val loss=1.1899, val macro-F1=0.3348 | 4.2s
Epoch 08/40 | train loss=1.2800, train macro-F1=0.3991 | val loss=1.2725, val macro-F1=0.3515 | 4.5s
Epoch 09/40 | train loss=1.2718, train macro-F1=0.4070 | val loss=1.2978, val macro-F1=0.2803 | 4.4s
Early

In [58]:

import torch
from sklearn.metrics import classification_report, confusion_matrix

checkpoint_path = CHECKPOINT_DIR / "cnn_B_balanced_seed42.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

model_B = CompactRespiratoryCNN(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

model_B.load_state_dict(checkpoint["model_state_dict"])
model_B.eval()

validation_loader = make_loader(
    "B", "validation", batch_size=BATCH_SIZE, num_workers=0
)

y_true_B, y_pred_B = [], []

with torch.no_grad():
    for batch in validation_loader:
        waveforms = batch["waveform"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        logits = model_B(waveforms)
        predictions = logits.argmax(dim=1)

        y_true_B.extend(labels.cpu().numpy().tolist())
        y_pred_B.extend(predictions.cpu().numpy().tolist())

print("========== EXPERIMENT B BALANCED VALIDATION REPORT ==========")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Saved best validation macro-F1:",
      round(checkpoint["best_val_macro_f1"], 4))

print("\nClassification report:")
print(classification_report(
    y_true_B,
    y_pred_B,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print("Class order:", CLASS_NAMES)
print(confusion_matrix(
    y_true_B, y_pred_B,
    labels=list(range(len(CLASS_NAMES)))
))

print("\nValidation-only evaluation; test set untouched.")


========== EXPERIMENT B BALANCED VALIDATION REPORT ==========
Checkpoint epoch: 2
Saved best validation macro-F1: 0.4082

Classification report:
              precision    recall  f1-score   support

      Normal     0.7885    0.7580    0.7729       595
    Crackles     0.2937    0.2295    0.2577       183
     Wheezes     0.5082    0.5299    0.5188       117
        Both     0.0519    0.2105    0.0833        19

    accuracy                         0.6116       914
   macro avg     0.4106    0.4320    0.4082       914
weighted avg     0.6382    0.6116    0.6229       914

Confusion matrix (rows=true, columns=predicted):
Class order: ['Normal', 'Crackles', 'Wheezes', 'Both']
[[451  83  37  24]
 [ 80  42  18  43]
 [ 32  17  62   6]
 [  9   1   5   4]]

Validation-only evaluation; test set untouched.


In [59]:

baseline_C_balanced = train_model(
    experiment="C",
    strategy="balanced",
    seed=42,
    max_epochs=40,
    patience=7
)

print("\n========== EXPERIMENT C BALANCED-SAMPLER SUMMARY ==========")
print("Best epoch:", baseline_C_balanced["best_epoch"])
print(
    "Best validation macro-F1:",
    round(baseline_C_balanced["best_val_macro_f1"], 4)
)
print("Checkpoint:", baseline_C_balanced["checkpoint_path"])


Starting experiment=C, strategy=balanced, train rows=43,591, sampled epoch size=4,423
Epoch 01/40 | train loss=1.3782, train macro-F1=0.2880 | val loss=1.3705, val macro-F1=0.2123 | 5.3s
Epoch 02/40 | train loss=1.3487, train macro-F1=0.3318 | val loss=1.4751, val macro-F1=0.1671 | 4.4s
Epoch 03/40 | train loss=1.3114, train macro-F1=0.3756 | val loss=1.2353, val macro-F1=0.3169 | 4.2s
Epoch 04/40 | train loss=1.3037, train macro-F1=0.3775 | val loss=1.2779, val macro-F1=0.3110 | 4.1s
Epoch 05/40 | train loss=1.2790, train macro-F1=0.3907 | val loss=1.4277, val macro-F1=0.2511 | 4.2s
Epoch 06/40 | train loss=1.2713, train macro-F1=0.4002 | val loss=1.2820, val macro-F1=0.3123 | 4.4s
Epoch 07/40 | train loss=1.2402, train macro-F1=0.4214 | val loss=1.4627, val macro-F1=0.2074 | 4.2s
Epoch 08/40 | train loss=1.2396, train macro-F1=0.4323 | val loss=1.2113, val macro-F1=0.3676 | 4.2s
Epoch 09/40 | train loss=1.2259, train macro-F1=0.4448 | val loss=1.4675, val macro-F1=0.1554 | 4.0s
Epoch

In [60]:

import torch
from sklearn.metrics import classification_report, confusion_matrix

checkpoint_path = CHECKPOINT_DIR / "cnn_C_balanced_seed42.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

model_C = CompactRespiratoryCNN(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

model_C.load_state_dict(checkpoint["model_state_dict"])
model_C.eval()

validation_loader = make_loader(
    "C", "validation", batch_size=BATCH_SIZE, num_workers=0
)

y_true_C, y_pred_C = [], []

with torch.no_grad():
    for batch in validation_loader:
        waveforms = batch["waveform"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        logits = model_C(waveforms)
        predictions = logits.argmax(dim=1)

        y_true_C.extend(labels.cpu().numpy().tolist())
        y_pred_C.extend(predictions.cpu().numpy().tolist())

print("========== EXPERIMENT C BALANCED VALIDATION REPORT ==========")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Saved best validation macro-F1:",
      round(checkpoint["best_val_macro_f1"], 4))

print("\nClassification report:")
print(classification_report(
    y_true_C,
    y_pred_C,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print("Class order:", CLASS_NAMES)
print(confusion_matrix(
    y_true_C,
    y_pred_C,
    labels=list(range(len(CLASS_NAMES)))
))

print("\nValidation-only evaluation; test set untouched.")


========== EXPERIMENT C BALANCED VALIDATION REPORT ==========
Checkpoint epoch: 15
Saved best validation macro-F1: 0.3906

Classification report:
              precision    recall  f1-score   support

      Normal     0.7643    0.5176    0.6172       595
    Crackles     0.2689    0.6011    0.3716       183
     Wheezes     0.5111    0.3932    0.4444       117
        Both     0.1667    0.1053    0.1290        19

    accuracy                         0.5098       914
   macro avg     0.4277    0.4043    0.3906       914
weighted avg     0.6203    0.5098    0.5358       914

Confusion matrix (rows=true, columns=predicted):
Class order: ['Normal', 'Crackles', 'Wheezes', 'Both']
[[308 252  32   3]
 [ 65 110   8   0]
 [ 24  40  46   7]
 [  6   7   4   2]]

Validation-only evaluation; test set untouched.


In [61]:

import pandas as pd

seed7_results = []

for experiment in ["A", "B", "C"]:
    result = train_model(
        experiment=experiment,
        strategy="balanced",
        seed=7,
        max_epochs=40,
        patience=7
    )

    seed7_results.append({
        "experiment": experiment,
        "strategy": "balanced",
        "seed": 7,
        "best_epoch": result["best_epoch"],
        "best_val_macro_f1": result["best_val_macro_f1"],
        "checkpoint_path": result["checkpoint_path"]
    })

seed7_summary = pd.DataFrame(seed7_results)

print("\n========== SEED 7 SUMMARY ==========")
print(seed7_summary.to_string(index=False))
print("\nValidation-only runs; test set untouched.")


Starting experiment=A, strategy=balanced, train rows=4,423, sampled epoch size=4,423
Epoch 01/40 | train loss=1.3262, train macro-F1=0.3389 | val loss=1.2881, val macro-F1=0.2284 | 5.2s
Epoch 02/40 | train loss=1.2849, train macro-F1=0.3805 | val loss=1.2425, val macro-F1=0.2458 | 4.5s
Epoch 03/40 | train loss=1.2615, train macro-F1=0.3985 | val loss=1.1716, val macro-F1=0.2716 | 3.9s
Epoch 04/40 | train loss=1.2358, train macro-F1=0.4200 | val loss=1.2949, val macro-F1=0.2291 | 3.8s
Epoch 05/40 | train loss=1.2123, train macro-F1=0.4391 | val loss=1.7645, val macro-F1=0.2058 | 3.8s
Epoch 06/40 | train loss=1.1966, train macro-F1=0.4592 | val loss=1.1592, val macro-F1=0.2948 | 3.7s
Epoch 07/40 | train loss=1.1692, train macro-F1=0.4672 | val loss=1.1440, val macro-F1=0.3245 | 3.7s
Epoch 08/40 | train loss=1.1554, train macro-F1=0.4898 | val loss=1.3000, val macro-F1=0.2818 | 3.6s
Epoch 09/40 | train loss=1.1460, train macro-F1=0.4911 | val loss=1.2741, val macro-F1=0.2660 | 4.1s
Epoch 

In [62]:

seed21_results = []

for experiment in ["A", "B", "C"]:
    result = train_model(
        experiment=experiment,
        strategy="balanced",
        seed=21,
        max_epochs=40,
        patience=7
    )

    seed21_results.append({
        "experiment": experiment,
        "strategy": "balanced",
        "seed": 21,
        "best_epoch": result["best_epoch"],
        "best_val_macro_f1": result["best_val_macro_f1"],
        "checkpoint_path": result["checkpoint_path"]
    })

seed21_summary = pd.DataFrame(seed21_results)

print("\n========== SEED 21 SUMMARY ==========")
print(seed21_summary.to_string(index=False))
print("\nValidation-only runs; test set untouched.")


Starting experiment=A, strategy=balanced, train rows=4,423, sampled epoch size=4,423
Epoch 01/40 | train loss=1.3359, train macro-F1=0.3310 | val loss=1.1687, val macro-F1=0.2834 | 4.9s
Epoch 02/40 | train loss=1.2928, train macro-F1=0.3762 | val loss=1.1499, val macro-F1=0.3167 | 3.9s
Epoch 03/40 | train loss=1.2578, train macro-F1=0.4028 | val loss=1.1593, val macro-F1=0.3133 | 3.8s
Epoch 04/40 | train loss=1.2330, train macro-F1=0.4205 | val loss=1.1759, val macro-F1=0.2528 | 3.8s
Epoch 05/40 | train loss=1.2116, train macro-F1=0.4437 | val loss=1.3147, val macro-F1=0.2863 | 4.0s
Epoch 06/40 | train loss=1.1659, train macro-F1=0.4768 | val loss=1.2853, val macro-F1=0.2675 | 3.9s
Epoch 07/40 | train loss=1.1733, train macro-F1=0.4746 | val loss=1.9244, val macro-F1=0.1434 | 3.8s
Epoch 08/40 | train loss=1.1595, train macro-F1=0.4725 | val loss=1.5872, val macro-F1=0.2205 | 3.9s
Epoch 09/40 | train loss=1.1320, train macro-F1=0.5040 | val loss=1.2419, val macro-F1=0.2571 | 3.7s
Early 

In [63]:
import pandas as pd

seed_results = pd.DataFrame([
    {"experiment": "A", "seed": 42, "macro_f1": 0.410600},
    {"experiment": "A", "seed": 7,  "macro_f1": 0.368010},
    {"experiment": "A", "seed": 21, "macro_f1": 0.316734},

    {"experiment": "B", "seed": 42, "macro_f1": 0.408200},
    {"experiment": "B", "seed": 7,  "macro_f1": 0.400103},
    {"experiment": "B", "seed": 21, "macro_f1": 0.393780},

    {"experiment": "C", "seed": 42, "macro_f1": 0.390600},
    {"experiment": "C", "seed": 7,  "macro_f1": 0.375684},
    {"experiment": "C", "seed": 21, "macro_f1": 0.340430},
])

seed_summary = (
    seed_results.groupby("experiment")["macro_f1"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

seed_summary["mean_std"] = seed_summary.apply(
    lambda r: f'{r["mean"]:.4f} ± {r["std"]:.4f}', axis=1
)

print("========== THREE-SEED VALIDATION SUMMARY ==========")
print(seed_summary.to_string(index=False))

print("\nIndividual seed results:")
print(
    seed_results.pivot(
        index="experiment", columns="seed", values="macro_f1"
    ).round(4)
)

print("\nAll values are validation macro-F1. Test set remains untouched.")

========== THREE-SEED VALIDATION SUMMARY ==========
experiment     mean      std      min    max        mean_std
         A 0.365115 0.047000 0.316734 0.4106 0.3651 ± 0.0470
         B 0.400694 0.007228 0.393780 0.4082 0.4007 ± 0.0072
         C 0.368905 0.025763 0.340430 0.3906 0.3689 ± 0.0258

Individual seed results:
seed            7       21      42
experiment                        
A           0.3680  0.3167  0.4106
B           0.4001  0.3938  0.4082
C           0.3757  0.3404  0.3906

All values are validation macro-F1. Test set remains untouched.


In [65]:
# Inspect the actual batch structure returned by the validation loader

batch = next(iter(validation_loader))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    print("Batch keys:", list(batch.keys()))

    for key, value in batch.items():
        if hasattr(value, "shape"):
            print(f"{key}: type={type(value).__name__}, shape={value.shape}")
        else:
            print(f"{key}: type={type(value).__name__}, value={str(value)[:150]}")

elif isinstance(batch, (tuple, list)):
    print("Number of elements:", len(batch))

    for i, value in enumerate(batch):
        if hasattr(value, "shape"):
            print(f"batch[{i}]: type={type(value).__name__}, shape={value.shape}")
        else:
            print(f"batch[{i}]: type={type(value).__name__}, value={str(value)[:150]}")

Batch type: <class 'dict'>
Batch keys: ['waveform', 'label', 'dataset', 'sample_id', 'subject_id', 'split', 'annotation_granularity', 'label_strength', 'was_cropped', 'was_padded']
waveform: type=Tensor, shape=torch.Size([32, 1, 12000])
label: type=Tensor, shape=torch.Size([32])
dataset: type=list, value=['ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBHI', 'ICBH
sample_id: type=list, value=['/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/audio/cycles/109/1b1_Al_001.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classificat
subject_id: type=list, value=['109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '109', '1
split: type=list, value=['validation', 'validation', 'validation', 'validation', 'validation', 'validation', 'validation', 'validation', 'validation', 'validation', 'validati
annotatio

In [67]:
# Fix the report column name and summarize per-class results.
# The previous cell may have stopped after processing Seed 7.

# If the first seed was already added, continue with any missing seeds.
completed_seeds = sorted({
    row["seed"] for row in all_reports
})

print("Seeds already recorded:", completed_seeds)

for seed in SEEDS_TO_CHECK:
    if seed in completed_seeds:
        continue

    checkpoint_path = (
        PROJECT_ROOT / "models" / "phenotype_cnn"
        / f"cnn_B_balanced_seed{seed}.pt"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )
    state_dict = (
        checkpoint["model_state_dict"]
        if isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
        else checkpoint
    )

    model = CompactRespiratoryCNN().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in validation_loader:
            waveforms = batch["waveform"].to(DEVICE)
            labels = batch["label"]
            predictions = model(waveforms).argmax(dim=1)

            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(predictions.cpu().numpy().tolist())

    report = classification_report(
        y_true, y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    for class_name in CLASS_NAMES:
        all_reports.append({
            "seed": seed,
            "class": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1": report[class_name]["f1-score"],
            "support": report[class_name]["support"]
        })

    all_confusion_matrices[seed] = confusion_matrix(
        y_true, y_pred, labels=[0, 1, 2, 3]
    )

per_class_results = pd.DataFrame(all_reports)

print("\n========== PER-CLASS SUMMARY ACROSS SEEDS ==========")
print(
    per_class_results.groupby("class")[
        ["precision", "recall", "f1"]
    ].agg(["mean", "std"]).round(4)
)

for seed in SEEDS_TO_CHECK:
    print(f"\n========== SEED {seed} CONFUSION MATRIX ==========")
    print(pd.DataFrame(
        all_confusion_matrices[seed],
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ))

print("\nValidation only; test set remains untouched.")

Seeds already recorded: []

========== PER-CLASS SUMMARY ACROSS SEEDS ==========
         precision          recall              f1        
              mean     std    mean     std    mean     std
class                                                     
Both        0.1253  0.0679  0.2807  0.1215  0.1694  0.0874
Crackles    0.3157  0.0191  0.4062  0.1559  0.3480  0.0788
Normal      0.7748  0.0168  0.7272  0.0423  0.7497  0.0235
Wheezes     0.4752  0.0677  0.2963  0.2136  0.3357  0.1699

========== SEED 7 CONFUSION MATRIX ==========
          Normal  Crackles  Wheezes  Both
Normal       404       150       33     8
Crackles      74        96       10     3
Wheezes       35        39       29    14
Both           5         9        1     4

========== SEED 21 CONFUSION MATRIX ==========
          Normal  Crackles  Wheezes  Both
Normal       443       129       10    13
Crackles      83        85        2    13
Wheezes       53        42       13     9
Both           7         4       

In [69]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score

CLASS_NAMES = ["Normal", "Crackles", "Wheezes", "Both"]
EXPERIMENTS = ["A", "B", "C"]
SEEDS = [7, 21, 42]

validation_loader = make_loader(
    experiment="B",
    split="validation",
    batch_size=32,
    num_workers=0
)

comparison_rows = []

for experiment in EXPERIMENTS:
    for seed in SEEDS:
        checkpoint_path = (
            PROJECT_ROOT / "models" / "phenotype_cnn"
            / f"cnn_{experiment}_balanced_seed{seed}.pt"
        )

        if not checkpoint_path.exists():
            raise FileNotFoundError(
                f"Missing checkpoint: {checkpoint_path}"
            )

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False
        )

        state_dict = (
            checkpoint["model_state_dict"]
            if isinstance(checkpoint, dict)
            and "model_state_dict" in checkpoint
            else checkpoint
        )

        model = CompactRespiratoryCNN().to(DEVICE)
        model.load_state_dict(state_dict)
        model.eval()

        y_true, y_pred = [], []

        with torch.no_grad():
            for batch in validation_loader:
                waveforms = batch["waveform"].to(DEVICE)
                labels = batch["label"]

                predictions = model(waveforms).argmax(dim=1)

                y_true.extend(labels.cpu().numpy().tolist())
                y_pred.extend(predictions.cpu().numpy().tolist())

        report = classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0
        )

        row = {
            "experiment": experiment,
            "seed": seed,
            "macro_f1": f1_score(
                y_true, y_pred, average="macro", zero_division=0
            )
        }

        for class_name in CLASS_NAMES:
            row[f"{class_name}_F1"] = report[class_name]["f1-score"]
            row[f"{class_name}_recall"] = report[class_name]["recall"]

        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

print("========== ALL RUNS: VALIDATION RESULTS ==========")
print(comparison_df.round(4).to_string(index=False))

metric_columns = [
    "macro_f1",
    "Normal_F1",
    "Crackles_F1",
    "Wheezes_F1",
    "Both_F1"
]

summary = comparison_df.groupby("experiment")[metric_columns].agg(
    ["mean", "std"]
)

print("\n========== MEAN ± STANDARD DEVIATION ==========")
print(summary.round(4).to_string())

print("\nValidation only. Test set remains untouched.")

========== ALL RUNS: VALIDATION RESULTS ==========
experiment  seed  macro_f1  Normal_F1  Normal_recall  Crackles_F1  Crackles_recall  Wheezes_F1  Wheezes_recall  Both_F1  Both_recall
         A     7    0.3680     0.7404         0.7092       0.3205           0.2951      0.4112          0.5641   0.0000       0.0000
         A    21    0.3167     0.7822         0.8118       0.2531           0.1694      0.2170          0.1966   0.0147       0.0526
         A    42    0.4106     0.7022         0.6538       0.4110           0.4098      0.4601          0.6410   0.0690       0.0526
         B     7    0.4001     0.7260         0.6790       0.4025           0.5246      0.3053          0.2479   0.1667       0.2105
         B    21    0.3938     0.7502         0.7445       0.3837           0.4645      0.1831          0.1111   0.2581       0.4211
         B    42    0.4082     0.7729         0.7580       0.2577           0.2295      0.5188          0.5299   0.0833       0.2105
         C     7  